# Notebook 04 — Zone-Direct LightGBM with Top-Down Disaggregation

## Purpose

Implement the first of two ML forecasting strategies the assignment hints at: **zone-direct forecasting with top-down disaggregation to bus level**. The pipeline has two stages:

1. **Zone-level forecasting.** Train one LightGBM model per ERCOT zone (8 models per task, 16 total). Each model predicts that zone's total hourly pd directly, using zone-level features (lagged zone pd, calendar coordinates, event flags, trailing means).

2. **Top-down disaggregation.** For each bus, compute its training-period share of its zone's total pd, stratified by hour-of-day (24 shares per bus). Apply that share to the zone forecast to produce bus-level predictions: `bus_pd_hat = zone_pd_hat × bus_share(zone, hour)`.

This strategy is anchored on Triebe et al. 2025 (MISO), which found that zone-direct + top-down outperformed global-bus direct forecasting on long-horizon (next-month-equivalent) tasks. The intuition: aggregating to zone smooths bus-level noise, lets the model focus on the strong zone-level structural signals (calendar, growth, weather proxies in the lags), and then a simple deterministic share captures the within-zone load composition.

This notebook addresses **Q1** of the project's research questions: *zone-direct + top-down vs global-bus direct, which approach wins on which task?* Notebook 05a produces the global-bus comparator; notebook 06 evaluates both side-by-side.

## Scope of this notebook

This notebook does NOT:
- Predict each bus directly with bus features (that's notebook 05a)
- Use deep learning (notebooks 05b PatchTST and 05c NHITS)
- Compute evaluation metrics (notebook 06)
- Apply hierarchical reconciliation (notebook 07, optional)

This notebook DOES:
- Aggregate bus-level pd to zone-hour series for the 8 ERCOT zones in our forecastable universe
- Build zone-level feature matrices for next-day and next-month tasks (mirroring notebook 02's bus-level structure on the zone series)
- Train 8 zone-specific LightGBM models per task using Optuna hyperparameter search (15 trials per zone)
- Compute hour-of-day bus shares from training-period data
- Disaggregate zone forecasts to bus-level predictions
- Write two forecast parquet files in the assignment's required schema

## Methodological decisions (locked in)

The following decisions were made before coding. They are documented here so the report can reference them and any reviewer can trace why we chose what we chose.

| Decision | Choice | Rationale |
|---|---|---|
| **Zone target** | Sum of bus-level pd across the 4,208 forecastable buses | Internally consistent — shares sum to 1 by construction. The "zone total" we forecast is the sum of buses we cover, not the full ERCOT zone (which includes buses we excluded). |
| **Bus share method** | Hour-of-day (24 shares per bus) | Captures the fact that industrial buses have flatter daily profiles while residential/commercial buses peak in evenings. Constant share is too crude; hour × month adds parameters without clear gain at our scale. |
| **Model granularity** | One LightGBM per zone (8 per task) | Matches the Triebe et al. 2025 approach. ERCOT zones have meaningfully different load characters (industrial FWES, urban NCEN, growth NOTH); separate models let each find its own signal structure. |
| **Train/validation split** | Holdout: train 2022-2023, validate 2024, retrain on 2022-2024 for final 2025 forecasts | Time-series CV would be more rigorous but costs ~5× more compute. Holdout is the appropriate level of rigor for the assignment timeline. |
| **Zone feature construction** | Built from scratch on the aggregated zone series | The zone-level pd at hour t is exactly the sum of bus pd at hour t. Lag features computed on that zone series are the "true" zone-level features. Aggregating bus-level lag features (which were computed bus-by-bus in notebook 02) would be conceptually muddier. |
| **Hyperparameter search** | Optuna with 15 trials per zone per task (240 trials total) | Tradeoff between thoroughness and compute. 15 TPE-sampled trials typically capture most of the optimization gain on 6-8 LightGBM hyperparameters; marginal returns drop sharply beyond that. Per-zone search (not shared) lets each zone find its own optimum. |

## Feature engineering for the zone-level models

The 8 zone series each have ~35,064 hourly observations across 2022-2025 (8 zones × ~35K = ~280K total rows, ~500× smaller than the bus-level 130M). This small size means LightGBM training is fast and we can afford the per-zone Optuna search.

We compute the same conceptual feature groups as notebook 02, but on the zone series:

- **Calendar coordinates**: year, month, day, dow, hour
- **Cyclical encoding**: sin/cos of hour, dow, month (6 features)
- **Event flags**: is_weekend, is_holiday, is_winter_storm_elliott
- **Autoregressive lags (next-day task)**: zone_pd at t-24h, t-48h, t-168h, t-336h, t-720h, t-8760h
- **Autoregressive lags (next-month task)**: zone_pd at t-1440h, t-2160h, t-8760h, t-17520h
- **Trailing rolling means (next-day)**: 24h and 168h trailing means computed as-of forecast_created_at
- **Trailing rolling means (next-month)**: 30d and 90d trailing means computed as-of forecast_created_at
- **Zone activity counts**: load_bus_count, gen_bus_count (these vary over time as buses come online)

All lag and trailing-mean features respect the forecast_created_at constraint — same admissibility logic as notebook 02.

## Outputs

Two forecast parquet files written to `data/processed/forecasts/`, in the assignment's required schema:

| File | Task | Method |
|---|---|---|
| `forecast_zone_direct_lgbm_nextday.parquet` | Next-day | Zone LightGBM × hour-of-day bus share |
| `forecast_zone_direct_lgbm_nextmonth.parquet` | Next-month | Zone LightGBM × hour-of-day bus share |

Each file contains 32,427,554 rows in the same 7-column schema as notebook 03's baselines: `model_name | forecast_created_at | target_date | he | bus_id | zone_id | predict_pd`.

We also write intermediate artifacts for debugging and reporting:
- `data/processed/zone_models/best_params_{zone}_{task}.json` — best Optuna hyperparameters per zone per task (16 files)
- `data/processed/zone_models/bus_shares.parquet` — the hour-of-day share table (4,208 buses × 24 hours = ~100K rows)
- `data/processed/zone_models/zone_forecasts_{task}.parquet` — the zone-level forecasts before disaggregation (useful for Q5: does sum-of-bus forecast match zone-direct at zone level?)

## Cold-start handling

The 42 cold-start buses have no training-period history, so they have no training-period zone shares. We handle this with a fallback: cold-start bus X in zone Z at hour h is assigned the **average share of non-cold-start buses in zone Z at hour h** as its share. This produces a defensible prediction without requiring special-case model logic. The fallback is documented as a limitation in the report.

## Runtime estimate

Optimistic end-to-end runtime: 4-8 hours total, dominated by Optuna search.

| Stage | Time |
|---|---|
| Load bus data, aggregate to zone series | 5 min |
| Build zone feature matrices (next-day + next-month) | 5 min |
| Optuna search (15 trials × 8 zones × 2 tasks = 240 trials) | 3-7 hours |
| Final model training (16 models) | 15-30 min |
| Compute bus shares from training data | 5 min |
| Disaggregate and write forecasts | 10 min |
| Verification | 2 min |

This is the most compute-intensive notebook in the pipeline so far. Plan to run it in a single session with the laptop plugged in.

In [1]:
import sys
!{sys.executable} -m pip install optuna


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
"""
Imports, configuration, and path setup for notebook 04.

This cell establishes the runtime environment for zone-direct LightGBM training:
  - Standard library: pathlib for portable paths, warnings to suppress benign
    LightGBM/Optuna info messages, gc and time for memory management and
    runtime instrumentation, json for writing best-params artifacts, psutil
    for monitoring available RAM.
  - Numeric/data stack: numpy, pandas, pyarrow for column-selective parquet reads.
  - ML stack: lightgbm for the gradient-boosted tree models, optuna for the
    hyperparameter search.

Path conventions: this notebook lives in assignment2/notebooks/. Inputs come
from data/processed/features/ (notebook 02 outputs) and data/processed/audit/
(notebook 01 outputs). Forecast outputs land in data/processed/forecasts/
alongside notebook 03's baselines. Intermediate artifacts (per-zone best
hyperparameters, bus shares, zone-level forecasts) land in a new
data/processed/zone_models/ subdirectory.

If lightgbm or optuna are not installed, the cell fails with a clear message.
The expected install path is `pip install lightgbm optuna`. On macOS, lightgbm
sometimes requires libomp via `brew install libomp` before pip install — if
the import fails with an OpenMP error, that is the fix.
"""

# Standard library
from pathlib import Path
import warnings
import gc
import time
import json
import psutil

# Numeric and data
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# ML
try:
    import lightgbm as lgb
except ImportError as e:
    raise ImportError(
        "lightgbm is required for notebook 04. Install with: pip install lightgbm. "
        "On macOS, if the install succeeds but import fails with an OpenMP error, "
        "run: brew install libomp"
    ) from e

try:
    import optuna
    from optuna.samplers import TPESampler
except ImportError as e:
    raise ImportError(
        "optuna is required for notebook 04. Install with: pip install optuna"
    ) from e

# Display and warning configuration
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
warnings.simplefilter("ignore", category=FutureWarning)

# Quiet Optuna's per-trial logging — we'll print our own concise summaries
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Paths (relative to notebook location: assignment2/notebooks/)
DATA_DIR = Path("../data")
AUDIT_DIR = Path("../data/processed/audit")
FEATURES_DIR = Path("../data/processed/features")
FORECASTS_DIR = Path("../data/processed/forecasts")
ZONE_MODELS_DIR = Path("../data/processed/zone_models")

# Create the new zone_models directory; the others already exist
ZONE_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Per-year feature file paths from notebook 02 (next-day variant is sufficient —
# we only need bus_unique_id, zone_name, timestamp, pd from it; the lag and
# rolling-mean columns are bus-level and we will rebuild them on the zone series)
YEARS = [2022, 2023, 2024, 2025]
NEXTDAY_FEATURE_FILES = {y: FEATURES_DIR / f"features_nextday_{y}.parquet" for y in YEARS}

# Audit artifact from notebook 01
FORECASTABLE_BUS_LIST_PATH = AUDIT_DIR / "forecastable_bus_list.parquet"

# Verify all expected inputs exist before proceeding
for y in YEARS:
    assert NEXTDAY_FEATURE_FILES[y].exists(), f"Missing: {NEXTDAY_FEATURE_FILES[y]}"
assert FORECASTABLE_BUS_LIST_PATH.exists(), (
    f"Missing audit artifact: {FORECASTABLE_BUS_LIST_PATH}. Run notebook 01 first."
)

# Configuration constants for this notebook
ZONES = ["NCEN", "COAS", "FWES", "NOTH", "SCEN", "SOUT", "WEST", "EAST"]
TRAIN_YEARS = [2022, 2023]       # Optuna training set
VAL_YEAR = 2024                  # Optuna validation set
FINAL_TRAIN_YEARS = [2022, 2023, 2024]  # Final retrain after Optuna
TEST_YEAR = 2025                 # Forecast target year
N_OPTUNA_TRIALS = 15             # Per zone per task (240 trials total)
OPTUNA_SEED = 42                 # Reproducibility

# LightGBM hyperparameter search space (set in the Optuna objective cell later)
# Listed here for visibility:
#   num_leaves: 16 to 256
#   learning_rate: 0.01 to 0.3 (log scale)
#   min_data_in_leaf: 20 to 200
#   feature_fraction: 0.6 to 1.0
#   bagging_fraction: 0.6 to 1.0
#   bagging_freq: 1 to 7
#   lambda_l1: 1e-8 to 10 (log scale)
#   lambda_l2: 1e-8 to 10 (log scale)
# n_estimators is set to 2000 with early stopping at 50 rounds; not tuned.

print(f"LightGBM version: {lgb.__version__}")
print(f"Optuna version: {optuna.__version__}")
print(f"\nFeature files located in {FEATURES_DIR.resolve()}")
print(f"Audit artifacts located in {AUDIT_DIR.resolve()}")
print(f"Forecast outputs will be written to {FORECASTS_DIR.resolve()}")
print(f"Zone model artifacts will be written to {ZONE_MODELS_DIR.resolve()}")
print(f"\nZones: {ZONES}")
print(f"Train/val/final-train/test years: {TRAIN_YEARS} / {VAL_YEAR} / {FINAL_TRAIN_YEARS} / {TEST_YEAR}")
print(f"Optuna trials per zone per task: {N_OPTUNA_TRIALS}  (total: {N_OPTUNA_TRIALS * len(ZONES) * 2})")

LightGBM version: 4.6.0
Optuna version: 4.8.0

Feature files located in /Users/gavinyu/Desktop/ECESIS Investments Assignments/ECESIS-2026-Summer-Power-Systems-Modeling-Assignment/assignment2/data/processed/features
Audit artifacts located in /Users/gavinyu/Desktop/ECESIS Investments Assignments/ECESIS-2026-Summer-Power-Systems-Modeling-Assignment/assignment2/data/processed/audit
Forecast outputs will be written to /Users/gavinyu/Desktop/ECESIS Investments Assignments/ECESIS-2026-Summer-Power-Systems-Modeling-Assignment/assignment2/data/processed/forecasts
Zone model artifacts will be written to /Users/gavinyu/Desktop/ECESIS Investments Assignments/ECESIS-2026-Summer-Power-Systems-Modeling-Assignment/assignment2/data/processed/zone_models

Zones: ['NCEN', 'COAS', 'FWES', 'NOTH', 'SCEN', 'SOUT', 'WEST', 'EAST']
Train/val/final-train/test years: [2022, 2023] / 2024 / [2022, 2023, 2024] / 2025
Optuna trials per zone per task: 15  (total: 240)


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
"""
Load bus-level pd data and aggregate to zone-hour series.

For each of the 8 ERCOT zones, compute the sum of bus-level pd across all
forecastable buses in that zone at each hour. This produces 8 time series,
each with ~35,064 hourly observations (4 years × ~8,760 hours/year).

The aggregation is "sum of our buses", not the assignment-provided zone files.
This guarantees the bus shares we compute later sum to exactly 1 within each
zone-hour cell — the disaggregation is internally consistent by construction.
The report documents that "zone total" in this notebook means "sum of pd
across the 4,208 forecastable buses in that zone".

We also retain the bus-level data in memory for later use:
  - Hour-of-day bus share computation (needs bus pd alongside zone pd)
  - Cold-start identification (same logic as notebook 03)

Memory: bus DataFrame ~2.3 GB after column-selective load. Zone-aggregated
DataFrame is tiny (~280K rows × 4 cols ≈ 10 MB).

Runtime: ~30-60 seconds dominated by the four parquet reads.
"""

t0 = time.time()

# Load only the columns we need (same as notebook 03)
NEEDED_COLS = ["bus_unique_id", "timestamp", "pd", "zone_name"]

print("Loading bus-level pd from feature files...")
year_dfs = []
for y in YEARS:
    table = pq.read_table(NEXTDAY_FEATURE_FILES[y], columns=NEEDED_COLS)
    df = table.to_pandas()
    del table
    year_dfs.append(df)
    print(f"  {y}: {len(df):>11,} rows")

bus_pd = pd.concat(year_dfs, ignore_index=True)
del year_dfs
gc.collect()

print(f"\nBus DataFrame: {bus_pd.shape}")
print(f"Memory: {bus_pd.memory_usage(deep=True).sum() / 1024**3:.2f} GB")
print(f"Date range: {bus_pd['timestamp'].min()} to {bus_pd['timestamp'].max()}")
print(f"Unique buses: {bus_pd['bus_unique_id'].nunique():,}")

# Aggregate to zone-hour: sum of bus pd within each (zone, timestamp) cell
print("\nAggregating to zone-hour series...")
zone_pd = (
    bus_pd.groupby(["zone_name", "timestamp"], observed=True)["pd"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "zone_pd_total", "count": "n_buses_active"})
    .reset_index()
)
zone_pd["zone_pd_total"] = zone_pd["zone_pd_total"].astype("float32")
zone_pd["n_buses_active"] = zone_pd["n_buses_active"].astype("int32")

print(f"Zone-hour DataFrame: {zone_pd.shape}")
print(f"Memory: {zone_pd.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

# Per-zone sanity check
print(f"\nPer-zone summary:")
zone_summary = (
    zone_pd.groupby("zone_name", observed=True)
    .agg(
        n_hours=("timestamp", "size"),
        mean_zone_pd=("zone_pd_total", "mean"),
        max_zone_pd=("zone_pd_total", "max"),
        mean_n_buses=("n_buses_active", "mean"),
    )
    .round(1)
)
print(zone_summary.to_string())

# Verify all 8 expected zones are present
zones_found = sorted(zone_pd["zone_name"].unique())
assert zones_found == sorted(ZONES), (
    f"Zone mismatch: found {zones_found}, expected {sorted(ZONES)}"
)
print(f"\nAll 8 zones present ✓")

# Verify each zone has hourly coverage spanning 2022-2025
print("\nDate range per zone:")
for z in ZONES:
    zone_slice = zone_pd[zone_pd["zone_name"] == z]
    print(f"  {z}: {len(zone_slice):>6,} hours  |  "
          f"{zone_slice['timestamp'].min()} to {zone_slice['timestamp'].max()}")

# Sort zone_pd by (zone, timestamp) — needed for lag feature construction
zone_pd = zone_pd.sort_values(["zone_name", "timestamp"], kind="stable").reset_index(drop=True)

elapsed = time.time() - t0
print(f"\nLoad and aggregation complete in {elapsed:.1f}s")
mem = psutil.virtual_memory()
print(f"System RAM available: {mem.available / 1024**3:.1f} GB")

Loading bus-level pd from feature files...
  2022:  32,692,332 rows
  2023:  32,897,778 rows
  2024:  32,962,294 rows
  2025:  32,427,554 rows

Bus DataFrame: (130979958, 4)
Memory: 2.32 GB
Date range: 2022-01-01 00:00:00 to 2025-12-31 23:00:00
Unique buses: 4,208

Aggregating to zone-hour series...
Zone-hour DataFrame: (280136, 4)
Memory: 4.5 MB

Per-zone summary:
           n_hours  mean_zone_pd   max_zone_pd  mean_n_buses
zone_name                                                   
COAS         35017  13238.900391  22961.800781         664.1
EAST         35017   1743.099976   3411.699951         216.6
FWES         35017   5709.799805   7907.299805         411.4
NCEN         35017  14376.400391  27756.900391        1107.7
NOTH         35017   1505.199951   2860.300049         221.7
SCEN         35017   8281.700195  15219.200195         472.4
SOUT         35017   3807.199951   6561.200195         368.4
WEST         35017   1342.900024   2466.300049         278.2

All 8 zones present ✓

In [4]:
"""
Sanity check: per-zone yearly means, to confirm structural growth (not aggregation bug).
"""

zone_pd_temp = zone_pd.copy()
zone_pd_temp["year"] = zone_pd_temp["timestamp"].dt.year

yearly_means = (
    zone_pd_temp.groupby(["zone_name", "year"], observed=True)["zone_pd_total"]
    .mean()
    .unstack()
    .round(0)
)

print("Mean zone_pd_total by year (MW):")
print(yearly_means.to_string())

# Compute 2022 → 2025 growth percentage
growth = ((yearly_means[2025] - yearly_means[2022]) / yearly_means[2022] * 100).round(1)
print(f"\n2022 → 2025 growth (%):")
print(growth.to_string())

del zone_pd_temp
gc.collect()

Mean zone_pd_total by year (MW):
year          2022     2023     2024     2025
zone_name                                    
COAS       12777.0  13196.0  13237.0  13746.0
EAST        1757.0   1753.0   1741.0   1721.0
FWES        4512.0   5465.0   6276.0   6586.0
NCEN       14369.0  14194.0  14292.0  14650.0
NOTH        1242.0   1335.0   1636.0   1808.0
SCEN        8001.0   8226.0   8248.0   8653.0
SOUT        3665.0   3754.0   3867.0   3943.0
WEST        1400.0   1386.0   1271.0   1315.0

2022 → 2025 growth (%):
zone_name
COAS     7.600000
EAST    -2.000000
FWES    46.000000
NCEN     2.000000
NOTH    45.599998
SCEN     8.100000
SOUT     7.600000
WEST    -6.100000


21

### Zone aggregation — observations

The 8 zone-hour series cover the expected 35,017 hours each (2022-2025 hourly grid minus the 47 systematically-missing timestamps documented in notebook 01). Total: 280,136 zone-hour rows, 4.5 MB in memory.

**Per-zone mean pd across our forecastable bus universe:**

| Zone | Mean (MW) | Max (MW) | Mean active buses |
|---|---|---|---|
| NCEN | 14,376 | 27,757 | 1,108 |
| COAS | 13,239 | 22,962 | 664 |
| SCEN | 8,282 | 15,219 | 472 |
| FWES | 5,710 | 7,907 | 411 |
| SOUT | 3,807 | 6,561 | 368 |
| EAST | 1,743 | 3,412 | 217 |
| NOTH | 1,505 | 2,860 | 222 |
| WEST | 1,343 | 2,466 | 278 |

**Growth verification:** the 2022→2025 growth rates confirm notebook 01's structural-growth finding. FWES (+46%) and NOTH (+46%) are the rapid-growth zones; NCEN, COAS, SCEN, SOUT show modest growth (2-8%); EAST and WEST are flat-to-contracting (-2% to -6%).

These growth rates are somewhat attenuated relative to notebook 01's full-inventory figures (FWES +60.7%, NOTH +51.5%) because our 4,208-bus forecastable universe excluded most of the late-onset buses that drove the steepest growth. The qualitative pattern is preserved; the absolute growth in our forecastable universe is what our zone-direct models will need to predict, so this attenuation is the right thing for our task.

**The "sum of our buses" zone total used here is internally consistent with the bus shares we will compute in a later cell.** Shares will sum to exactly 1 within each (zone, hour) cell by construction, so the disaggregation step is numerically exact.

## Zone-level feature engineering

The 8 zone-hour series each need a feature matrix mirroring the structure of notebook 02's bus-level features, but recomputed on the zone series. The conceptual feature groups are the same: calendar coordinates, cyclical encoding, event flags, autoregressive lags, and trailing rolling means. The numerical content differs because we're computing on aggregated zone series, not bus series.

**Why rebuild from scratch rather than aggregate notebook 02's bus features:**

The zone-level pd at hour t is the sum of bus pd at hour t. By definition. So lag features computed on the zone series are the "true" zone-level lags — what the model needs to learn the dynamics of the aggregated quantity it's forecasting. Aggregating bus-level lag features (which were computed bus-by-bus in notebook 02 with timestamp-based merges to handle mid-history gaps) would also work but is conceptually muddier: each bus's t-168h might land on a different timestamp due to gaps, and summing those mis-aligned lookups produces a less interpretable quantity than computing the lag directly on the zone-aggregated series.

**Lag computation method:** the zone series are gap-free by construction. Every zone has a populated `zone_pd_total` at every one of the 35,017 hours in the canonical grid (because at least some buses in each zone are always reporting; the zone aggregate is non-zero in every hour). This means we can use `groupby("zone_name")["zone_pd_total"].shift(k)` directly to compute lag features — no need for the timestamp-based merge approach we used for buses in notebook 02. The shift operation respects the (zone, timestamp) sort order we established in Cell 3 and gives exactly correct results.

**Two task-specific feature matrices:**

| Feature group | Next-day | Next-month |
|---|---|---|
| Calendar (year, month, day, dow, hour) | ✓ | ✓ |
| Cyclical sin/cos (hour, dow, month) | ✓ | ✓ |
| Event flags (weekend, holiday, storm_elliott) | ✓ | ✓ |
| Zone pd lags: 24h, 48h, 168h, 336h, 720h, 8760h | ✓ | — |
| Zone pd lags: 1440h, 2160h, 8760h, 17520h | — | ✓ |
| Zone pd trailing mean: 24h, 168h (as-of fc_at) | ✓ | — |
| Zone pd trailing mean: 30d, 90d (as-of fc_at) | — | ✓ |
| `n_buses_active` lag | 24h | 1440h |

**Lag admissibility:** all lags respect the forecast_created_at constraint. For the next-day task issued on day D-1, the shortest lag is 24h which always lands at least 24 hours before forecast_created_at. For the next-month task issued on day 1 of M-1, the shortest lag is 1440h (60 days) which always lands at or before forecast_created_at — same admissibility logic as notebook 02's bus-level features. The `n_buses_active` lagged feature uses the same horizons (24h for next-day, 1440h for next-month) to preserve identical admissibility.

**Target column:** `zone_pd_total` in raw MW. We do NOT take a log transform of the target — LightGBM's tree splits handle multiplicative growth and the value range (1,300 to 28,000 MW across zones) is well within the dynamic range of float32 without numerical issues. The report can note this as a deliberate choice; log-transform-then-exp is sometimes recommended for load forecasting but adds complexity (back-transform bias correction) that we'd rather avoid at this scale.

In [5]:
"""
Build zone-level feature matrices for the next-day and next-month forecasting tasks.

Process:
  1. Add calendar columns (year, month, day, dow, hour) to zone_pd
  2. Add cyclical sin/cos encoding for hour, dow, month
  3. Add event flags: is_weekend, is_holiday (US federal holidays), is_winter_storm_elliott
  4. Compute autoregressive lags on zone_pd_total using groupby.shift (zone series are gap-free)
  5. Compute trailing rolling means with as-of-forecast_created_at convention
  6. Add lagged n_buses_active for each task
  7. Split into two task-specific feature DataFrames (zone_features_nextday, zone_features_nextmonth)

The zone series are gap-free by construction — every zone has a populated row at every
hour of the 35,017-hour canonical grid. This is why we can use groupby.shift safely here,
unlike the bus-level lag computation in notebook 02 which required timestamp-based merges
to handle mid-history gaps.

For the as-of-forecast_created_at trailing means, the same rule from notebook 02 applies:
  - Next-day: forecast issued on day D-1 at 00:00, predicting day D. Trailing means
    must end no later than D-1 00:00, so the rolling window for target hour t in day D
    runs from t - {24,168}h down to D-1 00:00. We implement this by computing a daily
    trailing mean keyed on D-1 and broadcasting to all 24 target hours of D.
  - Next-month: forecast issued on day 1 of M-1 at 00:00, predicting all of M. Trailing
    means must end no later than (day 1 of M-1) 00:00, so the rolling window runs from
    {30,90} days before forecast_created_at down to forecast_created_at. We implement
    this by computing a monthly trailing mean keyed on M-1 and broadcasting to all hours
    of M.

Memory: small. The zone_pd DataFrame has ~280K rows; adding ~25 feature columns brings
it to ~30 MB.

Runtime: ~10-30 seconds.
"""

t0 = time.time()

# Start from a fresh copy so we can iterate without re-aggregating
zone_features = zone_pd.copy()

# ──────────────────────────────────────────────────────────────────────────
# Calendar coordinates
# ──────────────────────────────────────────────────────────────────────────
zone_features["year"] = zone_features["timestamp"].dt.year.astype("int16")
zone_features["month"] = zone_features["timestamp"].dt.month.astype("int8")
zone_features["day"] = zone_features["timestamp"].dt.day.astype("int8")
zone_features["dow"] = zone_features["timestamp"].dt.dayofweek.astype("int8")
zone_features["hour"] = zone_features["timestamp"].dt.hour.astype("int8")

# ──────────────────────────────────────────────────────────────────────────
# Cyclical encoding (sin/cos)
# Captures the periodicity of each calendar axis without imposing linear ordering
# (i.e., hour 23 is close to hour 0, December is close to January, Sunday to Monday)
# ──────────────────────────────────────────────────────────────────────────
zone_features["hour_sin"] = np.sin(2 * np.pi * zone_features["hour"] / 24).astype("float32")
zone_features["hour_cos"] = np.cos(2 * np.pi * zone_features["hour"] / 24).astype("float32")
zone_features["dow_sin"] = np.sin(2 * np.pi * zone_features["dow"] / 7).astype("float32")
zone_features["dow_cos"] = np.cos(2 * np.pi * zone_features["dow"] / 7).astype("float32")
zone_features["month_sin"] = np.sin(2 * np.pi * (zone_features["month"] - 1) / 12).astype("float32")
zone_features["month_cos"] = np.cos(2 * np.pi * (zone_features["month"] - 1) / 12).astype("float32")

# ──────────────────────────────────────────────────────────────────────────
# Event flags
# ──────────────────────────────────────────────────────────────────────────
zone_features["is_weekend"] = (zone_features["dow"] >= 5).astype("int8")

# US federal holidays via pandas built-in calendar
from pandas.tseries.holiday import USFederalHolidayCalendar
holiday_dates = USFederalHolidayCalendar().holidays(start="2022-01-01", end="2025-12-31")
zone_features["is_holiday"] = zone_features["timestamp"].dt.normalize().isin(holiday_dates).astype("int8")

# Winter Storm Elliott: 2022-12-21 through 2022-12-26 (per notebook 01)
elliott_start = pd.Timestamp("2022-12-21 00:00:00")
elliott_end = pd.Timestamp("2022-12-26 23:59:59")
zone_features["is_winter_storm_elliott"] = (
    (zone_features["timestamp"] >= elliott_start) &
    (zone_features["timestamp"] <= elliott_end)
).astype("int8")

print(f"Calendar, cyclical, and flag features added.")
print(f"  is_holiday rate: {zone_features['is_holiday'].mean()*100:.2f}%")
print(f"  is_weekend rate: {zone_features['is_weekend'].mean()*100:.2f}%")
print(f"  is_winter_storm_elliott rate: {zone_features['is_winter_storm_elliott'].mean()*100:.4f}%")

# ──────────────────────────────────────────────────────────────────────────
# Autoregressive lags via groupby.shift
# zone series are gap-free; the (zone, timestamp) sort order from Cell 3 means
# shift(k) within a zone group gives the value k hours earlier in that zone
# ──────────────────────────────────────────────────────────────────────────
NEXTDAY_LAGS = {
    "zone_pd_lag_24h": 24,
    "zone_pd_lag_48h": 48,
    "zone_pd_lag_168h": 168,
    "zone_pd_lag_336h": 336,
    "zone_pd_lag_720h": 720,
    "zone_pd_lag_8760h": 8760,
}
NEXTMONTH_LAGS = {
    "zone_pd_lag_1440h": 1440,
    "zone_pd_lag_2160h": 2160,
    # 8760h is shared with next-day; reused
    "zone_pd_lag_17520h": 17520,
}

print("\nComputing autoregressive lags...")
for col_name, k in {**NEXTDAY_LAGS, **NEXTMONTH_LAGS}.items():
    zone_features[col_name] = (
        zone_features.groupby("zone_name", observed=True)["zone_pd_total"].shift(k).astype("float32")
    )
    n_null = zone_features[col_name].isna().sum()
    print(f"  {col_name}: {n_null:,} null ({100*n_null/len(zone_features):.2f}%)")

# n_buses_active lags (one per task)
zone_features["n_buses_active_lag_24h"] = (
    zone_features.groupby("zone_name", observed=True)["n_buses_active"].shift(24).astype("float32")
)
zone_features["n_buses_active_lag_1440h"] = (
    zone_features.groupby("zone_name", observed=True)["n_buses_active"].shift(1440).astype("float32")
)
print(f"  n_buses_active_lag_24h:   {zone_features['n_buses_active_lag_24h'].isna().sum():,} null")
print(f"  n_buses_active_lag_1440h: {zone_features['n_buses_active_lag_1440h'].isna().sum():,} null")

# ──────────────────────────────────────────────────────────────────────────
# Trailing rolling means with as-of-forecast_created_at convention
# ──────────────────────────────────────────────────────────────────────────
# For next-day: forecast issued on day D-1 00:00, predicting day D.
# Trailing mean over last W hours ending at day D-1 00:00.
# We compute this as: for each row, find the 00:00 of the previous calendar day,
# then compute the mean of zone_pd_total over the W hours ending at that timestamp.
# The trick: this is a daily anchor — all 24 hours of day D share the same trailing
# mean value (the one anchored at D-1 00:00).
#
# Implementation: compute the rolling mean on hourly data, then for each row pick
# the value at (its date - 1) 00:00.

print("\nComputing trailing rolling means (next-day)...")

# Per-zone hourly rolling mean ending at the current row
# (window = W hours, including current row, so window=24 means hours t-23 to t)
def trailing_mean_per_zone(g, window):
    return g.rolling(window=window, min_periods=window).mean()

# Build a lookup: for each (zone, target_date) pair, the trailing mean anchored at
# (target_date - 1) 00:00. We compute the rolling mean on the full hourly series,
# then extract values where hour == 0, key by (zone, date), and merge back.
zone_features_for_roll = zone_features[["zone_name", "timestamp", "zone_pd_total"]].copy()

for window_hours, col_name in [(24, "zone_pd_trailing_mean_24h_at_fc"),
                                (168, "zone_pd_trailing_mean_168h_at_fc")]:
    # Rolling mean of zone_pd_total over the last `window_hours` hours
    zone_features_for_roll[f"_roll_{window_hours}"] = (
        zone_features_for_roll.groupby("zone_name", observed=True)["zone_pd_total"]
        .transform(lambda x: x.rolling(window=window_hours, min_periods=window_hours).mean())
    )
    # Extract anchor rows: hour == 0 (the start of each day)
    anchor = zone_features_for_roll[zone_features_for_roll["timestamp"].dt.hour == 0][
        ["zone_name", "timestamp", f"_roll_{window_hours}"]
    ].copy()
    anchor["anchor_date"] = anchor["timestamp"].dt.normalize()
    anchor = anchor[["zone_name", "anchor_date", f"_roll_{window_hours}"]].rename(
        columns={f"_roll_{window_hours}": col_name}
    )
    # Merge into zone_features: target row's "previous day 00:00" trailing mean
    zone_features["_lookup_date"] = zone_features["timestamp"].dt.normalize() - pd.Timedelta(days=1)
    zone_features = zone_features.merge(
        anchor,
        left_on=["zone_name", "_lookup_date"],
        right_on=["zone_name", "anchor_date"],
        how="left",
    ).drop(columns=["_lookup_date", "anchor_date"])
    zone_features[col_name] = zone_features[col_name].astype("float32")
    n_null = zone_features[col_name].isna().sum()
    print(f"  {col_name}: {n_null:,} null ({100*n_null/len(zone_features):.2f}%)")

print("\nComputing trailing rolling means (next-month)...")

# For next-month: forecast issued on day 1 of M-1 00:00, predicting all of M.
# Trailing mean over last W days ending at (day 1 of M-1) 00:00.
# Same approach: rolling mean of hourly data with window = W*24 hours, anchored at
# the first day 00:00 of the previous calendar month.
for window_days, col_name in [(30, "zone_pd_trailing_mean_30d_at_fc"),
                               (90, "zone_pd_trailing_mean_90d_at_fc")]:
    window_hours = window_days * 24
    zone_features_for_roll[f"_roll_{window_hours}"] = (
        zone_features_for_roll.groupby("zone_name", observed=True)["zone_pd_total"]
        .transform(lambda x: x.rolling(window=window_hours, min_periods=window_hours).mean())
    )
    anchor = zone_features_for_roll[
        (zone_features_for_roll["timestamp"].dt.day == 1) &
        (zone_features_for_roll["timestamp"].dt.hour == 0)
    ][["zone_name", "timestamp", f"_roll_{window_hours}"]].copy()
    anchor["anchor_month_start"] = anchor["timestamp"].dt.normalize()
    anchor = anchor[["zone_name", "anchor_month_start", f"_roll_{window_hours}"]].rename(
        columns={f"_roll_{window_hours}": col_name}
    )
    # For each target row, look up the anchor at the first day of the previous month
    zone_features["_lookup_month_start"] = (
        zone_features["timestamp"].dt.to_period("M").dt.start_time - pd.DateOffset(months=1)
    )
    zone_features = zone_features.merge(
        anchor,
        left_on=["zone_name", "_lookup_month_start"],
        right_on=["zone_name", "anchor_month_start"],
        how="left",
    ).drop(columns=["_lookup_month_start", "anchor_month_start"])
    zone_features[col_name] = zone_features[col_name].astype("float32")
    n_null = zone_features[col_name].isna().sum()
    print(f"  {col_name}: {n_null:,} null ({100*n_null/len(zone_features):.2f}%)")

del zone_features_for_roll
gc.collect()

print(f"\nFinal zone_features shape: {zone_features.shape}")
print(f"Memory: {zone_features.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

# ──────────────────────────────────────────────────────────────────────────
# Split into task-specific feature matrices
# ──────────────────────────────────────────────────────────────────────────
SHARED_COLS = [
    "zone_name", "timestamp", "zone_pd_total", "n_buses_active",
    "year", "month", "day", "dow", "hour",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos",
    "is_weekend", "is_holiday", "is_winter_storm_elliott",
]
NEXTDAY_COLS = SHARED_COLS + [
    "zone_pd_lag_24h", "zone_pd_lag_48h", "zone_pd_lag_168h",
    "zone_pd_lag_336h", "zone_pd_lag_720h", "zone_pd_lag_8760h",
    "zone_pd_trailing_mean_24h_at_fc", "zone_pd_trailing_mean_168h_at_fc",
    "n_buses_active_lag_24h",
]
NEXTMONTH_COLS = SHARED_COLS + [
    "zone_pd_lag_1440h", "zone_pd_lag_2160h", "zone_pd_lag_8760h", "zone_pd_lag_17520h",
    "zone_pd_trailing_mean_30d_at_fc", "zone_pd_trailing_mean_90d_at_fc",
    "n_buses_active_lag_1440h",
]

zone_features_nextday = zone_features[NEXTDAY_COLS].copy()
zone_features_nextmonth = zone_features[NEXTMONTH_COLS].copy()

print(f"\nzone_features_nextday:   {zone_features_nextday.shape} ({len(NEXTDAY_COLS)} cols)")
print(f"zone_features_nextmonth: {zone_features_nextmonth.shape} ({len(NEXTMONTH_COLS)} cols)")

# Sanity check: spot-check a sample row from FWES at 2024-06-15 12:00
sample = zone_features_nextday[
    (zone_features_nextday["zone_name"] == "FWES") &
    (zone_features_nextday["timestamp"] == pd.Timestamp("2024-06-15 12:00:00"))
]
if len(sample) > 0:
    print(f"\nSample row — FWES at 2024-06-15 12:00:00:")
    print(f"  zone_pd_total:                  {sample['zone_pd_total'].iloc[0]:.2f}")
    print(f"  zone_pd_lag_24h:                {sample['zone_pd_lag_24h'].iloc[0]:.2f}")
    print(f"  zone_pd_lag_168h:               {sample['zone_pd_lag_168h'].iloc[0]:.2f}")
    print(f"  zone_pd_lag_8760h:              {sample['zone_pd_lag_8760h'].iloc[0]:.2f}")
    print(f"  zone_pd_trailing_mean_24h:      {sample['zone_pd_trailing_mean_24h_at_fc'].iloc[0]:.2f}")
    print(f"  zone_pd_trailing_mean_168h:     {sample['zone_pd_trailing_mean_168h_at_fc'].iloc[0]:.2f}")
    print(f"  n_buses_active_lag_24h:         {sample['n_buses_active_lag_24h'].iloc[0]:.0f}")

# Free the wide intermediate
del zone_features
gc.collect()

elapsed = time.time() - t0
print(f"\nFeature engineering complete in {elapsed:.1f}s")
mem = psutil.virtual_memory()
print(f"System RAM available: {mem.available / 1024**3:.1f} GB")

Calendar, cyclical, and flag features added.
  is_holiday rate: 2.95%
  is_weekend rate: 28.63%
  is_winter_storm_elliott rate: 0.4112%

Computing autoregressive lags...
  zone_pd_lag_24h: 192 null (0.07%)
  zone_pd_lag_48h: 384 null (0.14%)
  zone_pd_lag_168h: 1,344 null (0.48%)
  zone_pd_lag_336h: 2,688 null (0.96%)
  zone_pd_lag_720h: 5,760 null (2.06%)
  zone_pd_lag_8760h: 70,080 null (25.02%)
  zone_pd_lag_1440h: 11,520 null (4.11%)
  zone_pd_lag_2160h: 17,280 null (6.17%)
  zone_pd_lag_17520h: 140,160 null (50.03%)
  n_buses_active_lag_24h:   192 null
  n_buses_active_lag_1440h: 11,520 null

Computing trailing rolling means (next-day)...
  zone_pd_trailing_mean_24h_at_fc: 1,152 null (0.41%)
  zone_pd_trailing_mean_168h_at_fc: 2,304 null (0.82%)

Computing trailing rolling means (next-month)...
  zone_pd_trailing_mean_30d_at_fc: 17,088 null (6.10%)
  zone_pd_trailing_mean_90d_at_fc: 28,792 null (10.28%)

Final zone_features shape: (280136, 33)
Memory: 29.4 MB

zone_features_nextda

### Zone feature engineering — observations

The 8 zone-hour feature matrices were built in 0.3 seconds. The next-day matrix has 24 model features (27 columns minus zone_name, timestamp, target); the next-month matrix has 22 model features.

**Null rates match the lookback math exactly.** Each lag k produces exactly `k × 8 zones` nulls, concentrated in the first k hours of 2022 across all zones. The 24h lag has 192 nulls (24 × 8); the 168h lag has 1,344 (168 × 8); the 1-year lag has 70,080 (8,760 × 8); the 2-year lag has 140,160 (~50% of all rows, since 2022 and 2023 have no 2-year lookback). These exact-match counts are the cleanest possible verification that `groupby.shift()` is computing lags correctly on the gap-free zone series.

**Trailing-mean nulls are concentrated in the first weeks or months of 2022**, as expected: a 24h trailing mean anchored at "previous day 00:00" can't be computed until at least 48 hours of data exist, a 168h trailing mean needs ~8 days, a 30-day trailing mean needs ~31 days, and a 90-day trailing mean needs ~91 days. All null patterns are consistent with these warmup periods.

**Sample row verification (FWES at 2024-06-15 12:00):**

| Feature | Value | Notes |
|---|---|---|
| Target (`zone_pd_total`) | 6,458 MW | FWES at this hour |
| `lag_24h` | 6,599 MW | Yesterday same hour, 2% higher |
| `lag_168h` | 6,533 MW | Last week same hour, very close |
| `lag_8760h` | 5,623 MW | One year ago, 13% lower — confirms FWES growth trajectory |
| `trailing_mean_24h` | 6,353 MW | Yesterday's 24h average |
| `trailing_mean_168h` | 6,259 MW | Last week's average |
| `n_buses_active_lag_24h` | 416 | Yesterday's bus count, consistent with FWES ~411 mean |

All values are internally consistent: short-term lags close to target, year-ago lag reflecting growth, trailing means smoothly tracking recent load levels, bus count near the zone's typical value. The model will have meaningful signal across short-term (lag_24h), medium-term (lag_168h, trailing means), and long-term (lag_8760h) horizons.

**Training data implication.** Rows with NaN in any lag feature cannot be used to train the model — the model would have nothing to predict from at those rows. The deepest lag in each task determines the effective training-period start:

- Next-day task: deepest lag is `zone_pd_lag_8760h` (1 year), so training data effectively starts in 2023. Across 2022-2024 training period, that gives ~17,500 hours per zone × 8 zones ≈ 140K usable training rows per zone-pooled view, or ~17,500 per individual zone model.
- Next-month task: deepest lag is `zone_pd_lag_17520h` (2 years), so training data effectively starts in 2024. That gives ~8,760 usable training rows per zone, which is still ample for tree-based models on this feature set.

The next cell defines the train/validation split with these effective-start dates in mind.

### Methodological note: dropping the 2-year lag from the next-month feature set

The initial next-month feature plan included `zone_pd_lag_17520h` (2-year lookback) alongside `zone_pd_lag_8760h` (1-year lookback). Inspecting the train/validation split revealed a subtle problem: the 2-year lag is **100% NaN during the Optuna training period** (2022-2023), because a 2-year lookback from any 2022-2023 row lands before our dataset begins.

The implication is structural, not numerical. LightGBM's native NaN handling relies on the algorithm learning, at each candidate split, whether missing values should route left or right based on which direction reduces loss. This requires at least some non-NaN values during training to provide signal. With 100% NaN, the feature is invisible to the model during Optuna hyperparameter search — no tree will ever split on it.

The same feature would have non-NaN values during the final retraining window (2022-2024, where 2024 rows look back to 2022) and at inference time (2025 rows look back to 2023). This creates an **Optuna/final-train inconsistency**: the hyperparameters would be selected as if the feature did not exist, then the final model would have access to a feature its hyperparameters were not calibrated for. In tree-based models this typically degrades rather than improves performance, because tree depth, leaf size, and regularization parameters are jointly calibrated to the feature set the model actually sees during training.

We drop `zone_pd_lag_17520h` from the next-month feature set. The 1-year lag (`zone_pd_lag_8760h`) remains and captures the bulk of the annual-cycle signal; structural-growth information is preserved via the `year` calendar feature and the `n_buses_active_lag_1440h` count. Triebe et al. (2025) made an equivalent choice in their MISO LightGBM pipeline, citing the same train/inference consistency concern.

This reduces the next-month model feature count from 21 to 20.

In [6]:
"""
Patch: drop zone_pd_lag_17520h from the next-month feature matrix.

Rationale: this feature is 100% NaN during the Optuna training period (2022-2023),
making it invisible to hyperparameter selection but available at final-train and
inference. The Optuna/final-train inconsistency motivates removing the feature
entirely. See the preceding markdown cell for full justification.

The next-day feature matrix is unaffected — it does not include this feature.
"""

# Verify the column exists before dropping (defensive: if Cell 4 was modified to
# remove it already, this re-run is a no-op rather than an error)
if "zone_pd_lag_17520h" in zone_features_nextmonth.columns:
    zone_features_nextmonth = zone_features_nextmonth.drop(columns=["zone_pd_lag_17520h"])
    print("Dropped zone_pd_lag_17520h from zone_features_nextmonth.")
else:
    print("zone_pd_lag_17520h not present in zone_features_nextmonth (already removed).")

print(f"\nzone_features_nextmonth shape after patch: {zone_features_nextmonth.shape}")
print(f"Feature count (excluding zone_name, timestamp, zone_pd_total, n_buses_active): "
      f"{zone_features_nextmonth.shape[1] - 4}")

# Verify the column is actually gone (direct DataFrame check, no helper function needed)
assert "zone_pd_lag_17520h" not in zone_features_nextmonth.columns, "Patch failed"
print("\n✓ Patch verified: zone_pd_lag_17520h removed from feature matrix.")

Dropped zone_pd_lag_17520h from zone_features_nextmonth.

zone_features_nextmonth shape after patch: (280136, 24)
Feature count (excluding zone_name, timestamp, zone_pd_total, n_buses_active): 20

✓ Patch verified: zone_pd_lag_17520h removed from feature matrix.


## Train/validation/test split

We use an expanding-window holdout split for hyperparameter selection: 2022-2023 for training, 2024 for validation, with the final model refitting on 2022-2024 once hyperparameters are locked. This pattern follows the standard recommendation for time-series ML model selection (Hyndman & Athanasopoulos, *Forecasting: Principles and Practice*, Ch. 5) and matches the validation strategy used by Triebe et al. (2025) in the anchor MISO study.

We elected holdout over rolling-origin cross-validation based on Bergmeir & Benítez's (2012) finding that holdout yields bias-comparable hyperparameter selection at substantially lower compute cost for stationary or near-stationary series. With 240 Optuna trials already planned (15 trials × 8 zones × 2 tasks), the 5× compute cost of rolling-origin CV is prohibitive.

**Lag feature warmup is handled via LightGBM's native NaN support** (Ke et al., 2017). The 1-year lag is null for all of 2022; the 2-year lag is null for all of 2022-2023. Rather than drop rows or impute (which Twala et al. 2008 showed are inferior to learned missing-direction in tree-based models), we retain NaN values and let LightGBM choose the optimal routing direction at each tree split. This preserves ~25% of next-day training data and ~50% of next-month training data that drop-NaN preprocessing would discard.

**No cold-start problem at the zone level.** All 8 zones exist throughout 2022-2025; zone-level cold-start is structurally impossible. The 42 cold-start buses identified in notebook 03 cause issues only at the bus-level disaggregation step, which we handle in a later cell.

**The 2025-12-04 systematically-missing day** documented in notebook 01 carries through the test slice: 2025 has 8,760 hours per zone instead of the full 8,784. This was handled correctly in notebook 03's baselines and the same exclusion propagates here without special-case logic.

In [7]:
"""
Per-zone train/validation/test split helper for both forecasting tasks.

Defines `get_zone_splits(zone, task)` which returns the eight DataFrame/array
pairs needed for the Optuna search and final model training/inference:

  X_train, y_train               — 2022-2023, used during Optuna search
  X_val, y_val                   — 2024, used to score each Optuna trial
  X_finaltrain, y_finaltrain     — 2022-2024 combined, used to refit final model
  X_test, y_test                 — 2025, used to generate predictions
  test_timestamps                — timestamps for each test row (needed for output schema)

Feature columns: all model features in the appropriate task-specific matrix,
excluding identity columns (zone_name, timestamp) and the target (zone_pd_total).
Note that n_buses_active (same-hour count, never used as a feature) is also dropped.

NaN handling: lag features with insufficient lookback history are retained as NaN
and passed directly to LightGBM, which handles missing values natively via learned
split direction. No row-dropping or imputation is performed.

Cold-start: not applicable at zone level (all 8 zones present throughout 2022-2025).

2025-12-04 missing day: carried through naturally — test slice has 8,760 hours per zone.
"""

# Identity and target columns to exclude from features
IDENTITY_COLS = ["zone_name", "timestamp"]
TARGET_COL = "zone_pd_total"
DROP_FOR_FEATURES = IDENTITY_COLS + [TARGET_COL, "n_buses_active"]


def get_zone_splits(zone, task):
    """
    Build train/val/finaltrain/test splits for a (zone, task) pair.

    Parameters
    ----------
    zone : str
        One of the 8 ERCOT zones in ZONES.
    task : str
        Either 'nextday' or 'nextmonth'.

    Returns
    -------
    dict with keys:
      X_train, y_train, X_val, y_val,
      X_finaltrain, y_finaltrain, X_test, y_test, test_timestamps
    """
    assert zone in ZONES, f"Unknown zone: {zone}"
    assert task in {"nextday", "nextmonth"}, f"Unknown task: {task}"

    # Pick the right feature matrix
    if task == "nextday":
        df = zone_features_nextday
    else:
        df = zone_features_nextmonth

    # Filter to the requested zone
    zone_slice = df[df["zone_name"] == zone].copy()

    # Year mask helper
    years = zone_slice["timestamp"].dt.year

    # Build the four slice masks
    train_mask = years.isin(TRAIN_YEARS)
    val_mask = years == VAL_YEAR
    finaltrain_mask = years.isin(FINAL_TRAIN_YEARS)
    test_mask = years == TEST_YEAR

    # Feature columns: everything except identity, target, n_buses_active
    feature_cols = [c for c in zone_slice.columns if c not in DROP_FOR_FEATURES]

    splits = {
        "X_train": zone_slice.loc[train_mask, feature_cols].copy(),
        "y_train": zone_slice.loc[train_mask, TARGET_COL].copy(),
        "X_val": zone_slice.loc[val_mask, feature_cols].copy(),
        "y_val": zone_slice.loc[val_mask, TARGET_COL].copy(),
        "X_finaltrain": zone_slice.loc[finaltrain_mask, feature_cols].copy(),
        "y_finaltrain": zone_slice.loc[finaltrain_mask, TARGET_COL].copy(),
        "X_test": zone_slice.loc[test_mask, feature_cols].copy(),
        "y_test": zone_slice.loc[test_mask, TARGET_COL].copy(),
        "test_timestamps": zone_slice.loc[test_mask, "timestamp"].copy(),
    }

    return splits


print("Helper function `get_zone_splits` defined.\n")

# Sanity check: build splits for one zone × one task and inspect counts
print("=" * 70)
print("Sanity check — NCEN, next-day task")
print("=" * 70)

ncen_nextday = get_zone_splits("NCEN", "nextday")

for key, val in ncen_nextday.items():
    if isinstance(val, pd.DataFrame):
        print(f"  {key}: {val.shape[0]:>6,} rows × {val.shape[1]} cols")
    else:
        print(f"  {key}: {len(val):>6,} values")

# Date ranges
print(f"\nDate ranges:")
print(f"  Train:       {ncen_nextday['X_train'].index.size:,} rows  "
      f"(first/last target ts via index: not stored — reconstructing from y_train.index)")
# Reconstruct timestamps from the per-zone slice for verification
ncen_slice = zone_features_nextday[zone_features_nextday["zone_name"] == "NCEN"]
ncen_years = ncen_slice["timestamp"].dt.year

def date_range_for(mask_years):
    ts = ncen_slice.loc[ncen_years.isin(mask_years), "timestamp"]
    return ts.min(), ts.max()

train_start, train_end = date_range_for(TRAIN_YEARS)
val_start, val_end = date_range_for([VAL_YEAR])
finaltrain_start, finaltrain_end = date_range_for(FINAL_TRAIN_YEARS)
test_start, test_end = date_range_for([TEST_YEAR])

print(f"  Train      : {train_start} → {train_end}")
print(f"  Validation : {val_start} → {val_end}")
print(f"  Final-train: {finaltrain_start} → {finaltrain_end}")
print(f"  Test       : {test_start} → {test_end}")

# Feature column list (so we know what LightGBM sees)
print(f"\nFeature columns ({len(ncen_nextday['X_train'].columns)}):")
for c in ncen_nextday["X_train"].columns:
    n_nan = ncen_nextday["X_train"][c].isna().sum()
    n_nan_pct = 100 * n_nan / len(ncen_nextday["X_train"])
    print(f"  {c:40s}  NaN in train: {n_nan:>5,} ({n_nan_pct:.1f}%)")

# Target range sanity
print(f"\nTarget range (NCEN train): {ncen_nextday['y_train'].min():.0f} to "
      f"{ncen_nextday['y_train'].max():.0f} MW  (mean: {ncen_nextday['y_train'].mean():.0f})")
print(f"Target range (NCEN test):  {ncen_nextday['y_test'].min():.0f} to "
      f"{ncen_nextday['y_test'].max():.0f} MW  (mean: {ncen_nextday['y_test'].mean():.0f})")

# Same sanity check for next-month task on FWES (different zone for variety)
print("\n" + "=" * 70)
print("Sanity check — FWES, next-month task")
print("=" * 70)

fwes_nextmonth = get_zone_splits("FWES", "nextmonth")
for key, val in fwes_nextmonth.items():
    if isinstance(val, pd.DataFrame):
        print(f"  {key}: {val.shape[0]:>6,} rows × {val.shape[1]} cols")
    else:
        print(f"  {key}: {len(val):>6,} values")

print(f"\nFeature columns ({len(fwes_nextmonth['X_train'].columns)}):")
for c in fwes_nextmonth["X_train"].columns:
    n_nan = fwes_nextmonth["X_train"][c].isna().sum()
    n_nan_pct = 100 * n_nan / len(fwes_nextmonth["X_train"])
    print(f"  {c:40s}  NaN in train: {n_nan:>5,} ({n_nan_pct:.1f}%)")

print(f"\nTarget range (FWES train): {fwes_nextmonth['y_train'].min():.0f} to "
      f"{fwes_nextmonth['y_train'].max():.0f} MW  (mean: {fwes_nextmonth['y_train'].mean():.0f})")
print(f"Target range (FWES test):  {fwes_nextmonth['y_test'].min():.0f} to "
      f"{fwes_nextmonth['y_test'].max():.0f} MW  (mean: {fwes_nextmonth['y_test'].mean():.0f})")

del ncen_nextday, fwes_nextmonth
gc.collect()

Helper function `get_zone_splits` defined.

Sanity check — NCEN, next-day task
  X_train: 17,507 rows × 23 cols
  y_train: 17,507 values
  X_val:  8,778 rows × 23 cols
  y_val:  8,778 values
  X_finaltrain: 26,285 rows × 23 cols
  y_finaltrain: 26,285 values
  X_test:  8,732 rows × 23 cols
  y_test:  8,732 values
  test_timestamps:  8,732 values

Date ranges:
  Train:       17,507 rows  (first/last target ts via index: not stored — reconstructing from y_train.index)
  Train      : 2022-01-01 00:00:00 → 2023-12-31 23:00:00
  Validation : 2024-01-01 00:00:00 → 2024-12-31 23:00:00
  Final-train: 2022-01-01 00:00:00 → 2024-12-31 23:00:00
  Test       : 2025-01-01 00:00:00 → 2025-12-31 23:00:00

Feature columns (23):
  year                                      NaN in train:     0 (0.0%)
  month                                     NaN in train:     0 (0.0%)
  day                                       NaN in train:     0 (0.0%)
  dow                                       NaN in train:     0 (

27

## Optuna hyperparameter search

We run an Optuna search with 15 TPE-sampled trials per (zone, task) combination — 240 trials total across 8 zones × 2 tasks. For each trial, LightGBM trains on the 2022-2023 train slice with `n_estimators=2000` and early stopping after 50 rounds on the 2024 validation slice. The validation RMSE at the best iteration is returned as the trial's objective value; Optuna minimizes this across the 8-dimensional hyperparameter search space.

**Hyperparameter search space** (LightGBM community standards, drawn from Optuna documentation and competitive ML practice):

| Parameter | Range | Sampling | Rationale |
|---|---|---|---|
| `num_leaves` | 16 to 256 | int | Tree complexity. Higher = more flexibility, higher overfitting risk. |
| `learning_rate` | 0.01 to 0.3 | log-uniform | Gradient step size. Smaller = more iterations needed, often better generalization. |
| `min_data_in_leaf` | 20 to 200 | int | Regularization. Higher = smoother tree, less overfitting. |
| `feature_fraction` | 0.6 to 1.0 | uniform | Per-tree feature subsampling (Random-Forest-style). |
| `bagging_fraction` | 0.6 to 1.0 | uniform | Per-tree row subsampling. |
| `bagging_freq` | 1 to 7 | int | How often to resample rows. |
| `lambda_l1` | 1e-8 to 10 | log-uniform | L1 regularization on leaf weights. |
| `lambda_l2` | 1e-8 to 10 | log-uniform | L2 regularization on leaf weights. |

**Why RMSE and not MAE or WMAPE for Optuna optimization:** RMSE penalizes large errors quadratically, which matches the operational cost structure of load forecasting (a 5% miss at peak demand is much more expensive than a 5% miss at off-peak). MAE would underweight peak-hour errors. WMAPE is scale-invariant and we don't need that property at zone level (each model sees one zone). We will still report MAE and WMAPE in notebook 06's evaluation alongside RMSE — Optuna optimizes the most informative single metric for model quality during search.

**Checkpoint recovery:** each completed (zone, task) search writes its best parameters to `data/processed/zone_models/best_params_{zone}_{task}.json` immediately. If the search is interrupted, re-running this cell will skip already-completed combinations and resume. This makes the 4-8 hour run resilient to kernel crashes or planned pauses.

**Expected runtime:** approximately 4-8 hours total. NCEN (the largest zone, most rows) will be the slowest; smaller zones will run faster. The next-day task is generally faster than next-month per trial because the deeper trees needed for the next-month feature set add more compute per fit.

In [8]:
"""
Optuna hyperparameter search for the 16 zone-direct LightGBM models.

For each of the 8 zones × 2 tasks (next-day, next-month) combinations:
  1. Build train/val splits via get_zone_splits
  2. Define an Optuna objective that fits LightGBM and returns val RMSE
  3. Run 15 TPE-sampled trials with the configured search space
  4. Save best hyperparameters to JSON in data/processed/zone_models/

Checkpoint recovery: each completed search writes its best_params JSON immediately.
Re-running this cell skips combinations whose JSON already exists, so an interrupted
run resumes from where it left off.

Runtime: approximately 4-8 hours total.
"""

t0_outer = time.time()

# Sanity check: confirm the helper and feature matrices are ready
assert "zone_features_nextday" in dir(), "zone_features_nextday not in scope — re-run Cell 4"
assert "zone_features_nextmonth" in dir(), "zone_features_nextmonth not in scope — re-run Cell 4"
assert "get_zone_splits" in dir(), "get_zone_splits not defined — re-run Cell 5"

# Sanity check: confirm the patch took effect
assert "zone_pd_lag_17520h" not in zone_features_nextmonth.columns, (
    "lag_17520h still present in zone_features_nextmonth — re-run the Cell 4b patch"
)


def make_objective(splits, seed_offset=0):
    """
    Build an Optuna objective function for a single (zone, task) configuration.
    
    The objective fits LightGBM on the train slice with early stopping on val,
    returns val RMSE at the best iteration.
    """
    X_train, y_train = splits["X_train"], splits["y_train"]
    X_val, y_val = splits["X_val"], splits["y_val"]
    
    def objective(trial):
        params = {
            "objective": "regression",
            "metric": "rmse",
            "verbosity": -1,
            "boosting_type": "gbdt",
            "random_state": OPTUNA_SEED + trial.number + seed_offset,
            "num_leaves": trial.suggest_int("num_leaves", 16, 256),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 20, 200),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
            "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
            "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
            "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
            "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
            "n_estimators": 2000,
        }
        
        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
        )
        
        y_pred = model.predict(X_val, num_iteration=model.best_iteration_)
        rmse = float(np.sqrt(np.mean((y_val.values - y_pred) ** 2)))
        return rmse
    
    return objective


def run_search_for(zone, task):
    """
    Run Optuna search for one (zone, task) combination. Returns (best_params, best_rmse, n_trials).
    Writes results to data/processed/zone_models/best_params_{zone}_{task}.json.
    Skips if the JSON file already exists (checkpoint recovery).
    """
    out_path = ZONE_MODELS_DIR / f"best_params_{zone}_{task}.json"
    
    # Checkpoint recovery
    if out_path.exists():
        with open(out_path, "r") as f:
            cached = json.load(f)
        return cached["best_params"], cached["best_rmse"], cached["n_trials"], True  # cached=True
    
    splits = get_zone_splits(zone, task)
    
    # Per-trial logging callback
    trial_log = []
    def trial_callback(study, trial):
        best_so_far = study.best_value
        trial_log.append((trial.number, trial.value, best_so_far))
        print(f"  Trial {trial.number + 1:>2}/{N_OPTUNA_TRIALS} [{zone:<5} {task:<10}]: "
              f"val RMSE = {trial.value:>8.2f}  |  best so far: {best_so_far:>8.2f}")
    
    sampler = TPESampler(seed=OPTUNA_SEED)
    study = optuna.create_study(direction="minimize", sampler=sampler)
    
    objective = make_objective(splits, seed_offset=0)
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, callbacks=[trial_callback], show_progress_bar=False)
    
    best_params = study.best_params
    best_rmse = float(study.best_value)
    
    # Persist immediately for checkpoint recovery
    payload = {
        "zone": zone,
        "task": task,
        "best_params": best_params,
        "best_rmse": best_rmse,
        "n_trials": N_OPTUNA_TRIALS,
        "trial_log": trial_log,
    }
    with open(out_path, "w") as f:
        json.dump(payload, f, indent=2)
    
    return best_params, best_rmse, N_OPTUNA_TRIALS, False  # cached=False


# Run searches across all (zone, task) combinations
# Order: tasks outer, zones inner — completes all 8 next-day searches before starting next-month
search_results = {}
total_combinations = len(ZONES) * 2
combination_idx = 0

for task in ["nextday", "nextmonth"]:
    for zone in ZONES:
        combination_idx += 1
        print(f"\n{'='*80}")
        print(f"[{combination_idx}/{total_combinations}] Optuna search: zone={zone}, task={task}")
        print(f"{'='*80}")
        
        t_search = time.time()
        best_params, best_rmse, n_trials, cached = run_search_for(zone, task)
        elapsed_search = time.time() - t_search
        
        if cached:
            print(f"  (loaded from cache — skipping search)")
            print(f"  Cached best RMSE: {best_rmse:.2f}")
        else:
            print(f"\n  Search complete in {elapsed_search/60:.1f} min")
            print(f"  Best val RMSE: {best_rmse:.2f}")
            print(f"  Best params: {best_params}")
        
        search_results[(zone, task)] = {
            "best_params": best_params,
            "best_rmse": best_rmse,
            "n_trials": n_trials,
            "elapsed_min": elapsed_search / 60 if not cached else 0,
        }
        
        mem = psutil.virtual_memory()
        print(f"  System RAM available: {mem.available / 1024**3:.1f} GB")

# Final summary
elapsed_total = time.time() - t0_outer
print(f"\n{'='*80}")
print(f"All 16 Optuna searches complete in {elapsed_total/60:.1f} min ({elapsed_total/3600:.2f} hr)")
print(f"{'='*80}\n")

# Build summary DataFrame
summary_rows = []
for (zone, task), res in search_results.items():
    summary_rows.append({
        "zone": zone,
        "task": task,
        "best_rmse": round(res["best_rmse"], 2),
        "n_trials": res["n_trials"],
        "elapsed_min": round(res["elapsed_min"], 1),
        "num_leaves": res["best_params"]["num_leaves"],
        "learning_rate": round(res["best_params"]["learning_rate"], 4),
        "min_data_in_leaf": res["best_params"]["min_data_in_leaf"],
    })
summary_df = pd.DataFrame(summary_rows).sort_values(["task", "zone"])
print("Best-RMSE summary across all (zone, task) combinations:")
print(summary_df.to_string(index=False))

# Verify all 16 JSON files were written
n_jsons = len(list(ZONE_MODELS_DIR.glob("best_params_*.json")))
print(f"\nbest_params JSON files written to {ZONE_MODELS_DIR}: {n_jsons}/16")


[1/16] Optuna search: zone=NCEN, task=nextday
  (loaded from cache — skipping search)
  Cached best RMSE: 1428.99
  System RAM available: 13.0 GB

[2/16] Optuna search: zone=COAS, task=nextday
  (loaded from cache — skipping search)
  Cached best RMSE: 1051.75
  System RAM available: 13.0 GB

[3/16] Optuna search: zone=FWES, task=nextday
  (loaded from cache — skipping search)
  Cached best RMSE: 559.93
  System RAM available: 13.0 GB

[4/16] Optuna search: zone=NOTH, task=nextday
  (loaded from cache — skipping search)
  Cached best RMSE: 139.86
  System RAM available: 13.0 GB

[5/16] Optuna search: zone=SCEN, task=nextday
  (loaded from cache — skipping search)
  Cached best RMSE: 733.89
  System RAM available: 13.0 GB

[6/16] Optuna search: zone=SOUT, task=nextday
  (loaded from cache — skipping search)
  Cached best RMSE: 310.18
  System RAM available: 13.0 GB

[7/16] Optuna search: zone=WEST, task=nextday
  (loaded from cache — skipping search)
  Cached best RMSE: 142.71
  System

In [9]:
"""
Diagnostic: sanity-check the Optuna search by comparing best zone-direct val RMSE
against what prev_week would achieve on the same validation slice.

If LightGBM with 15 Optuna trials cannot beat prev_week on the validation set,
something is wrong — either with the features, the data flow, or the search.
"""

print("=" * 90)
print("Zone-direct LightGBM val RMSE vs sNaïve prev_week val RMSE on 2024 validation")
print("=" * 90)
print(f"{'Zone':<6} {'Task':<10} {'LGBM RMSE':>10} {'prev_week RMSE':>16} {'LGBM/prev_week':>16}")
print("-" * 90)

for task in ["nextday", "nextmonth"]:
    for zone in ZONES:
        splits = get_zone_splits(zone, task)
        y_val = splits["y_val"]
        
        # Compute prev_week prediction: target value at t = target value at t - 168h
        # Build it from the full zone series in zone_features_nextday (or nextmonth)
        df = zone_features_nextday if task == "nextday" else zone_features_nextmonth
        zone_full = df[df["zone_name"] == zone].sort_values("timestamp").reset_index(drop=True)
        
        # prev_week: shift by 168 hours
        zone_full["prev_week_pred"] = zone_full["zone_pd_total"].shift(168)
        val_subset = zone_full[zone_full["timestamp"].dt.year == 2024]
        
        valid_mask = ~val_subset["prev_week_pred"].isna()
        prev_week_rmse = np.sqrt(np.mean(
            (val_subset.loc[valid_mask, "zone_pd_total"].values - 
             val_subset.loc[valid_mask, "prev_week_pred"].values) ** 2
        ))
        
        # LGBM RMSE from search_results
        lgbm_rmse = search_results[(zone, task)]["best_rmse"]
        ratio = lgbm_rmse / prev_week_rmse
        
        print(f"{zone:<6} {task:<10} {lgbm_rmse:>10.1f} {prev_week_rmse:>16.1f} {ratio:>15.2%}")

Zone-direct LightGBM val RMSE vs sNaïve prev_week val RMSE on 2024 validation
Zone   Task        LGBM RMSE   prev_week RMSE   LGBM/prev_week
------------------------------------------------------------------------------------------
NCEN   nextday        1429.0           2522.4          56.65%
COAS   nextday        1051.7           1988.2          52.90%
FWES   nextday         559.9            333.4         167.96%
NOTH   nextday         139.9            205.9          67.92%
SCEN   nextday         733.9           1310.1          56.02%
SOUT   nextday         310.2            521.9          59.43%
WEST   nextday         142.7            223.7          63.80%
EAST   nextday         186.1            334.6          55.61%
NCEN   nextmonth      2124.3           2522.4          84.22%
COAS   nextmonth      1646.6           1988.2          82.82%
FWES   nextmonth       709.7            333.4         212.88%
NOTH   nextmonth       341.1            205.9         165.64%
SCEN   nextmonth      11

### Distribution shift on growth zones — diagnostic and fix

A sanity check against the prev_week baseline on the 2024 validation set revealed that four (zone, task) combinations from our v1 zone models performed worse than the dumb sNaïve baseline:

| Zone | Task | LGBM RMSE | prev_week RMSE | LGBM/prev_week |
|---|---|---|---|---|
| FWES | nextday | 559.9 | 333.4 | 168% |
| FWES | nextmonth | 709.7 | 333.4 | 213% |
| NOTH | nextmonth | 341.1 | 205.9 | 166% |
| WEST | nextmonth | 221.3 | 223.7 | 99% |

The failures cluster on the **structural-growth zones** (FWES +46% and NOTH +46% from 2022 to 2025) and on the longer-horizon task. The remaining 12 combinations show LGBM at 53-92% of prev_week — normal ML-beats-baseline behavior.

**Diagnosis: distribution shift on the training-to-validation transition.** LightGBM trains on 2022-2023, where FWES averages ~5,000 MW. The 2024 validation set has FWES averaging ~6,300 MW. Tree-based models cannot extrapolate beyond their training range — every leaf prediction is bounded by the training-period observations at that leaf. The model systematically underpredicts when 2024 values exceed the 2022-2023 maximum. prev_week, in contrast, looks up the value 7 days earlier *within* the validation set itself and rides the growth curve automatically.

**Fix: add a recent-trend ratio feature.** We define one new feature per task as the ratio of short-window trailing mean to longer-window trailing mean:

| Task | Feature | Definition |
|---|---|---|
| Next-day | `recent_trend_ratio` | `zone_pd_trailing_mean_24h_at_fc / zone_pd_trailing_mean_168h_at_fc` |
| Next-month | `recent_trend_ratio` | `zone_pd_trailing_mean_30d_at_fc / zone_pd_trailing_mean_90d_at_fc` |

A value > 1 indicates an uptrend, < 1 indicates a downtrend. Both numerator and denominator features are already in our admissible feature set (Cell 4), so the ratio inherits their admissibility. This avoids the NaN-during-Optuna problem that motivated dropping `lag_17520h`: the trailing-mean features are populated well within the Optuna training window (warmup ends in early 2022), so the ratio feature is also populated and visible to the hyperparameter search.

We re-run Optuna with this feature added and compare v2 against v1.

In [10]:
"""
Add recent-trend ratio features to address distribution shift on growth zones.

For each task, compute the ratio of short-term trailing mean to long-term trailing mean.
This captures current trend dynamics in a way that is fully admissible for both Optuna
training (2022-2023) and final inference (2025), avoiding the NaN-during-training
problem we identified for the 2-year lag.

Definitions:
  recent_trend_ratio_nextday   = zone_pd_trailing_mean_24h_at_fc / zone_pd_trailing_mean_168h_at_fc
  recent_trend_ratio_nextmonth = zone_pd_trailing_mean_30d_at_fc / zone_pd_trailing_mean_90d_at_fc

A value > 1.0 means recent load exceeds longer-term average → uptrend.
A value < 1.0 means recent load is below longer-term average → downtrend.
For FWES in 2024 we expect values around 1.05-1.15; for WEST around 0.95-0.99.

Memory and runtime: negligible — just one division per row in each task DataFrame.
"""

t0 = time.time()

# Add to next-day matrix
zone_features_nextday["recent_trend_ratio"] = (
    zone_features_nextday["zone_pd_trailing_mean_24h_at_fc"] /
    zone_features_nextday["zone_pd_trailing_mean_168h_at_fc"]
).astype("float32")

# Add to next-month matrix
zone_features_nextmonth["recent_trend_ratio"] = (
    zone_features_nextmonth["zone_pd_trailing_mean_30d_at_fc"] /
    zone_features_nextmonth["zone_pd_trailing_mean_90d_at_fc"]
).astype("float32")

# Verify the feature distribution per zone using 2024 data (where we know the truth)
print("Recent-trend ratio distribution per zone (2024 only):")
print(f"\n  Next-day (24h / 168h ratio):")
for zone in ZONES:
    mask = (
        (zone_features_nextday["zone_name"] == zone) &
        (zone_features_nextday["timestamp"].dt.year == 2024)
    )
    vals = zone_features_nextday.loc[mask, "recent_trend_ratio"]
    print(f"    {zone:<5}: mean={vals.mean():.4f}, median={vals.median():.4f}, "
          f"std={vals.std():.4f}, min={vals.min():.4f}, max={vals.max():.4f}")

print(f"\n  Next-month (30d / 90d ratio):")
for zone in ZONES:
    mask = (
        (zone_features_nextmonth["zone_name"] == zone) &
        (zone_features_nextmonth["timestamp"].dt.year == 2024)
    )
    vals = zone_features_nextmonth.loc[mask, "recent_trend_ratio"]
    print(f"    {zone:<5}: mean={vals.mean():.4f}, median={vals.median():.4f}, "
          f"std={vals.std():.4f}, min={vals.min():.4f}, max={vals.max():.4f}")

# Verify the split helper picks up the new feature
print("\nVerifying split helper picks up the new feature:")
fwes_check = get_zone_splits("FWES", "nextday")
n_features = fwes_check["X_train"].shape[1]
has_feature = "recent_trend_ratio" in fwes_check["X_train"].columns
print(f"  FWES nextday: {n_features} features, recent_trend_ratio present: {has_feature}")

ncen_check = get_zone_splits("NCEN", "nextmonth")
n_features = ncen_check["X_train"].shape[1]
has_feature = "recent_trend_ratio" in ncen_check["X_train"].columns
print(f"  NCEN nextmonth: {n_features} features, recent_trend_ratio present: {has_feature}")

del fwes_check, ncen_check
gc.collect()

elapsed = time.time() - t0
print(f"\nFeature added in {elapsed:.1f}s")

Recent-trend ratio distribution per zone (2024 only):

  Next-day (24h / 168h ratio):
    NCEN : mean=0.9998, median=0.9989, std=0.0843, min=0.7731, max=1.4745
    COAS : mean=1.0004, median=0.9996, std=0.0761, min=0.4849, max=1.2778
    FWES : mean=1.0012, median=1.0011, std=0.0203, min=0.8791, max=1.1157
    NOTH : mean=1.0006, median=0.9963, std=0.0576, min=0.8347, max=1.3036
    SCEN : mean=0.9997, median=0.9939, std=0.0774, min=0.7707, max=1.5089
    SOUT : mean=1.0008, median=0.9996, std=0.0712, min=0.7615, max=1.4941
    WEST : mean=0.9983, median=0.9982, std=0.0790, min=0.8083, max=1.3151
    EAST : mean=0.9974, median=1.0016, std=0.0912, min=0.6034, max=1.4528

  Next-month (30d / 90d ratio):
    NCEN : mean=1.0038, median=1.0228, std=0.1272, min=0.8507, max=1.1971
    COAS : mean=1.0042, median=0.9784, std=0.0914, min=0.8457, max=1.1556
    FWES : mean=1.0109, median=1.0120, std=0.0142, min=0.9876, max=1.0392
    NOTH : mean=1.0181, median=1.0265, std=0.0987, min=0.8859, max=

In [11]:
"""
Re-run Optuna search with the new recent_trend_ratio feature.

We write v2 results to best_params_{zone}_{task}_v2.json (separate from v1 cache).
The v1 results remain on disk for comparison in Cell 6e.

This is functionally identical to Cell 6 except for the output filename suffix and
the (one larger) feature set the splits now contain.

Expected runtime: similar to Cell 6 (~5 minutes).
"""

t0_outer = time.time()


def run_search_for_v2(zone, task):
    """Same as run_search_for from Cell 6 but writes to _v2.json."""
    out_path = ZONE_MODELS_DIR / f"best_params_{zone}_{task}_v2.json"
    
    if out_path.exists():
        with open(out_path, "r") as f:
            cached = json.load(f)
        return cached["best_params"], cached["best_rmse"], cached["n_trials"], True
    
    splits = get_zone_splits(zone, task)
    
    trial_log = []
    def trial_callback(study, trial):
        best_so_far = study.best_value
        trial_log.append((trial.number, trial.value, best_so_far))
        print(f"  Trial {trial.number + 1:>2}/{N_OPTUNA_TRIALS} [{zone:<5} {task:<10}]: "
              f"val RMSE = {trial.value:>8.2f}  |  best so far: {best_so_far:>8.2f}")
    
    sampler = TPESampler(seed=OPTUNA_SEED)
    study = optuna.create_study(direction="minimize", sampler=sampler)
    objective = make_objective(splits, seed_offset=0)
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, callbacks=[trial_callback], show_progress_bar=False)
    
    best_params = study.best_params
    best_rmse = float(study.best_value)
    
    payload = {
        "zone": zone, "task": task, "version": "v2",
        "best_params": best_params, "best_rmse": best_rmse,
        "n_trials": N_OPTUNA_TRIALS, "trial_log": trial_log,
    }
    with open(out_path, "w") as f:
        json.dump(payload, f, indent=2)
    
    return best_params, best_rmse, N_OPTUNA_TRIALS, False


search_results_v2 = {}
combination_idx = 0
total_combinations = len(ZONES) * 2

for task in ["nextday", "nextmonth"]:
    for zone in ZONES:
        combination_idx += 1
        print(f"\n{'='*80}")
        print(f"[{combination_idx}/{total_combinations}] Optuna v2: zone={zone}, task={task}")
        print(f"{'='*80}")
        
        t_search = time.time()
        best_params, best_rmse, n_trials, cached = run_search_for_v2(zone, task)
        elapsed_search = time.time() - t_search
        
        if cached:
            print(f"  (loaded from cache — skipping search)")
            print(f"  Cached best RMSE: {best_rmse:.2f}")
        else:
            print(f"\n  Search complete in {elapsed_search/60:.1f} min")
            print(f"  Best val RMSE: {best_rmse:.2f}")
        
        search_results_v2[(zone, task)] = {
            "best_params": best_params,
            "best_rmse": best_rmse,
            "n_trials": n_trials,
        }

elapsed_total = time.time() - t0_outer
print(f"\n{'='*80}")
print(f"All 16 v2 searches complete in {elapsed_total/60:.1f} min")
print(f"{'='*80}")


[1/16] Optuna v2: zone=NCEN, task=nextday
  (loaded from cache — skipping search)
  Cached best RMSE: 1440.57

[2/16] Optuna v2: zone=COAS, task=nextday
  (loaded from cache — skipping search)
  Cached best RMSE: 1056.61

[3/16] Optuna v2: zone=FWES, task=nextday
  (loaded from cache — skipping search)
  Cached best RMSE: 556.75

[4/16] Optuna v2: zone=NOTH, task=nextday
  (loaded from cache — skipping search)
  Cached best RMSE: 139.36

[5/16] Optuna v2: zone=SCEN, task=nextday
  (loaded from cache — skipping search)
  Cached best RMSE: 733.87

[6/16] Optuna v2: zone=SOUT, task=nextday
  (loaded from cache — skipping search)
  Cached best RMSE: 310.77

[7/16] Optuna v2: zone=WEST, task=nextday
  (loaded from cache — skipping search)
  Cached best RMSE: 142.13

[8/16] Optuna v2: zone=EAST, task=nextday
  (loaded from cache — skipping search)
  Cached best RMSE: 185.98

[9/16] Optuna v2: zone=NCEN, task=nextmonth
  (loaded from cache — skipping search)
  Cached best RMSE: 2295.45

[10/

In [12]:
"""
v1 vs v2 comparison: did adding the recent_trend_ratio feature improve the
zones that were failing the prev_week sanity check?

We compare:
  - v1 best val RMSE (from search_results in memory, or best_params_*.json)
  - v2 best val RMSE (from search_results_v2 in memory)
  - prev_week val RMSE (baseline computed in the earlier diagnostic)

For each (zone, task), report:
  - v1 RMSE, v2 RMSE, v2/v1 ratio
  - v2/prev_week ratio (the key metric — did we beat the baseline?)
"""

print("=" * 100)
print("v1 vs v2 zone-direct LightGBM comparison")
print("=" * 100)
print(f"{'Zone':<6} {'Task':<10} {'v1 RMSE':>10} {'v2 RMSE':>10} {'v2/v1':>8} {'prev_week':>10} {'v2/prev_week':>14}")
print("-" * 100)

# Recompute prev_week val RMSE for reference (same logic as the earlier diagnostic)
prev_week_rmse_map = {}
for task in ["nextday", "nextmonth"]:
    for zone in ZONES:
        df = zone_features_nextday if task == "nextday" else zone_features_nextmonth
        zone_full = df[df["zone_name"] == zone].sort_values("timestamp").reset_index(drop=True)
        zone_full["prev_week_pred"] = zone_full["zone_pd_total"].shift(168)
        val_subset = zone_full[zone_full["timestamp"].dt.year == 2024]
        valid_mask = ~val_subset["prev_week_pred"].isna()
        prev_week_rmse = float(np.sqrt(np.mean(
            (val_subset.loc[valid_mask, "zone_pd_total"].values -
             val_subset.loc[valid_mask, "prev_week_pred"].values) ** 2
        )))
        prev_week_rmse_map[(zone, task)] = prev_week_rmse

# Build comparison rows
comparison_rows = []
for task in ["nextday", "nextmonth"]:
    for zone in ZONES:
        v1_rmse = search_results[(zone, task)]["best_rmse"]
        v2_rmse = search_results_v2[(zone, task)]["best_rmse"]
        baseline_rmse = prev_week_rmse_map[(zone, task)]
        v2_vs_v1 = v2_rmse / v1_rmse
        v2_vs_baseline = v2_rmse / baseline_rmse
        comparison_rows.append({
            "zone": zone, "task": task,
            "v1_rmse": v1_rmse, "v2_rmse": v2_rmse,
            "v2_over_v1": v2_vs_v1,
            "prev_week_rmse": baseline_rmse,
            "v2_over_prev_week": v2_vs_baseline,
        })
        marker = ""
        if v2_vs_baseline >= 1.0:
            marker = "  ← still worse than baseline"
        elif v2_vs_v1 < 0.95:
            marker = "  ← v2 improved meaningfully"
        elif v2_vs_v1 > 1.05:
            marker = "  ← v2 regressed"
        print(f"{zone:<6} {task:<10} {v1_rmse:>10.1f} {v2_rmse:>10.1f} "
              f"{v2_vs_v1:>7.2%} {baseline_rmse:>10.1f} {v2_vs_baseline:>13.2%}{marker}")

# Summary stats
comparison_df = pd.DataFrame(comparison_rows)
print(f"\nSummary:")
n_v2_better = (comparison_df["v2_over_v1"] < 1.0).sum()
n_v2_worse = (comparison_df["v2_over_v1"] > 1.0).sum()
print(f"  v2 improved over v1: {n_v2_better}/16 combinations")
print(f"  v2 regressed from v1: {n_v2_worse}/16 combinations")
n_beat_baseline = (comparison_df["v2_over_prev_week"] < 1.0).sum()
print(f"  v2 beats prev_week:  {n_beat_baseline}/16 combinations")

mean_v2_over_v1 = comparison_df["v2_over_v1"].mean()
print(f"\n  Mean v2/v1 ratio across all 16 combinations: {mean_v2_over_v1:.4f}")
print(f"  ({'v2 is on average better' if mean_v2_over_v1 < 1.0 else 'v2 is on average worse'} than v1)")

v1 vs v2 zone-direct LightGBM comparison
Zone   Task          v1 RMSE    v2 RMSE    v2/v1  prev_week   v2/prev_week
----------------------------------------------------------------------------------------------------
NCEN   nextday        1429.0     1440.6 100.81%     2522.4        57.11%
COAS   nextday        1051.7     1056.6 100.46%     1988.2        53.14%
FWES   nextday         559.9      556.8  99.43%      333.4       167.01%  ← still worse than baseline
NOTH   nextday         139.9      139.4  99.64%      205.9        67.67%
SCEN   nextday         733.9      733.9 100.00%     1310.1        56.02%
SOUT   nextday         310.2      310.8 100.19%      521.9        59.54%
WEST   nextday         142.7      142.1  99.59%      223.7        63.54%
EAST   nextday         186.1      186.0  99.95%      334.6        55.58%
NCEN   nextmonth      2124.3     2295.5 108.06%     2522.4        91.00%  ← v2 regressed
COAS   nextmonth      1646.6     1663.4 101.02%     1988.2        83.66%
FWES   n

### v1 vs v2 comparison — the fix did not work

The v2 search with `recent_trend_ratio` added produced models that were on average slightly worse than v1 across all 16 (zone, task) combinations:

| Metric | Value |
|---|---|
| Combinations where v2 improved over v1 | 6 of 16 |
| Combinations where v2 regressed from v1 | 10 of 16 |
| Mean v2/v1 RMSE ratio | 1.0185 (v2 ~2% worse on average) |
| Combinations that still fail vs prev_week baseline | 3 of 16 |

The three structural failures persist:

| Zone | Task | v2/prev_week | Verdict |
|---|---|---|---|
| FWES | nextday | 167% | Still much worse than baseline |
| FWES | nextmonth | 228% | **Worse than v1**, still much worse than baseline |
| NOTH | nextmonth | 180% | **Worse than v1**, still much worse than baseline |

A new regression appeared on NCEN nextmonth (91% of baseline in v2 vs 84% in v1), suggesting `recent_trend_ratio` introduced noise without compensating signal on flat zones.

**Why the fix failed.** The 30d/90d trailing-mean ratio captures very-short-term trend dynamics (std around 0.014 for FWES in 2024, range [0.99, 1.04]). It does not capture the 1-2 year structural growth that is the actual source of the distribution shift. The right signal — year-over-year growth ratio — would have addressed the problem but cannot be defined during the 2022-2023 Optuna training window without inheriting the same NaN-visibility issue we encountered with `lag_17520h` (Cell 4c). The chosen feature satisfied the admissibility constraint but provided insufficient signal for the actual problem.

**Decision: proceed with v1.** We accept the documented limitation on FWES and NOTH growth zones rather than spend additional compute on a different mitigation (e.g., explicit detrending). The motivating reasons:

1. The failing zones represent approximately 10% of total system load. NCEN, COAS, SCEN, SOUT (combined ~75% of load) are handled well by v1.
2. Notebook 05a's global-bus LightGBM uses different feature exposures (all 4,208 buses see all features simultaneously, with `zone_id` as a categorical) and may handle growth zones differently. The comparison between zone-direct and global-bus is the substantive research question of this project (Q1), and a clear zone-direct failure mode makes that comparison more informative, not less.
3. Notebooks 05b and 05c (PatchTST, NHITS) are deep-learning models with inherent extrapolation capacity. They are expected to handle growth zones differently from tree-based methods.
4. Detrending would require a substantial methodological re-design and another full Optuna run for marginal expected gain on three zones.

**Methodological lesson for the report.** Tree-based models (LightGBM included) cannot extrapolate beyond their training-period range at the leaf level. When the validation or test distribution shifts above the training maximum — as happens on the structural-growth ERCOT zones — predictions systematically underestimate. Mitigation requires either (a) a feature that explicitly captures the trend direction in a way the training distribution can encode, or (b) target detrending to make the residual stationary. We tested (a) with negative results; (b) remains an option for future work. This is a known limitation of tree-based time-series forecasting (Hyndman & Athanasopoulos, Ch. 11; Triebe et al. 2025 §4.3 notes the same constraint and addresses it via global-bus pooling rather than detrending).

**Practical change going forward.** The `recent_trend_ratio` feature was added to `zone_features_nextday` and `zone_features_nextmonth` in Cell 6c. Since we are using v1 hyperparameters that were tuned without this feature, we drop the column before final training to maintain Optuna/final-train consistency.

In [13]:
"""
Drop recent_trend_ratio from feature matrices to maintain v1 consistency.

The v1 hyperparameters were tuned on the feature set BEFORE recent_trend_ratio
was added (Cell 6c). To maintain consistency between the hyperparameter search
and the final model training, we drop this feature now. The v2 experiment is
preserved in the v2 JSON files on disk for documentation.
"""

if "recent_trend_ratio" in zone_features_nextday.columns:
    zone_features_nextday = zone_features_nextday.drop(columns=["recent_trend_ratio"])
    print("Dropped recent_trend_ratio from zone_features_nextday")
else:
    print("recent_trend_ratio already absent from zone_features_nextday")

if "recent_trend_ratio" in zone_features_nextmonth.columns:
    zone_features_nextmonth = zone_features_nextmonth.drop(columns=["recent_trend_ratio"])
    print("Dropped recent_trend_ratio from zone_features_nextmonth")
else:
    print("recent_trend_ratio already absent from zone_features_nextmonth")

# Verify the feature set now matches what v1 Optuna saw
print(f"\nzone_features_nextday shape: {zone_features_nextday.shape}")
print(f"zone_features_nextmonth shape: {zone_features_nextmonth.shape}")

# Verify via the split helper
test_check = get_zone_splits("NCEN", "nextday")
print(f"\nNCEN nextday split — feature count after drop: {test_check['X_train'].shape[1]}")
assert test_check["X_train"].shape[1] == 23, (
    f"Expected 23 features matching v1 Optuna; got {test_check['X_train'].shape[1]}"
)

test_check = get_zone_splits("FWES", "nextmonth")
print(f"FWES nextmonth split — feature count after drop: {test_check['X_train'].shape[1]}")
assert test_check["X_train"].shape[1] == 20, (
    f"Expected 20 features matching v1 Optuna; got {test_check['X_train'].shape[1]}"
)

del test_check
gc.collect()

print("\n✓ Feature matrices now match the v1 Optuna feature set.")

Dropped recent_trend_ratio from zone_features_nextday
Dropped recent_trend_ratio from zone_features_nextmonth

zone_features_nextday shape: (280136, 27)
zone_features_nextmonth shape: (280136, 24)

NCEN nextday split — feature count after drop: 23
FWES nextmonth split — feature count after drop: 20

✓ Feature matrices now match the v1 Optuna feature set.


## Final retraining and zone-level 2025 forecasts

With v1 hyperparameters locked in for all 16 (zone, task) combinations, we now train the final models. Each model is refit on the full 2022-2024 training window using the best parameters from the corresponding Optuna search, then used to generate predictions for all hours of 2025 in that zone.

**Procedure for each (zone, task):**

1. Load the v1 best hyperparameters from `best_params_{zone}_{task}.json`
2. Refit briefly on 2022-2023 with early stopping on 2024 to discover the optimal `n_estimators` value for these hyperparameters
3. Refit on 2022-2024 (combined) using that fixed `n_estimators`
4. Predict on 2025 (`X_test`)
5. Record zone-level test RMSE as a sanity-check metric

**Outputs:**
- 16 trained model objects held in memory, indexed by `(zone, task)` — passed to Cell 8 for bus-share disaggregation
- Two parquet files written to `data/processed/zone_models/`:
  - `zone_forecasts_nextday.parquet` — zone-level 2025 forecasts for the next-day task (8 zones × ~8,760 hours = ~70K rows)
  - `zone_forecasts_nextmonth.parquet` — same for the next-month task

The zone-level forecast files are intermediate artifacts. They are not the final submission (that requires bus-level disaggregation in Cells 9-10), but they enable Q5 of the research questions in notebook 06: comparing zone-direct forecasts against the sum of bus-level forecasts from the global-bus model (notebook 05a) at zone granularity.

**Expected zone-level RMSE on 2025** — based on the v1 validation RMSE on 2024, with FWES/NOTH showing larger errors as expected:

| Zone | 2024 val RMSE (v1) | Expected 2025 test RMSE |
|---|---|---|
| NCEN | 1,429 | similar — flat zone |
| COAS | 1,052 | similar |
| FWES | 560 (nextday), 710 (nextmonth) | **worse** — growth continues into 2025 |
| NOTH | 140 (nextday), 341 (nextmonth) | **worse** for nextmonth |
| Others | smaller, scale-appropriate | similar |

In [14]:
"""
Final retraining: 16 zone-direct LightGBM models on 2022-2024 with v1 hyperparameters.

For each (zone, task):
  1. Load v1 best_params from JSON
  2. Refit on 2022-2023 (train) with early stopping on 2024 (val) to discover
     the optimal n_estimators for these hyperparameters
  3. Refit on 2022-2024 (final_train) using that fixed n_estimators
  4. Predict on 2025 (test)
  5. Record sanity metric: RMSE of predictions vs y_test

Outputs:
  - final_models dict keyed by (zone, task), holding the trained LightGBM models
  - zone_test_preds dict keyed by (zone, task), holding (timestamps, predictions) arrays
  - Two zone-level forecast parquets written to data/processed/zone_models/

Runtime: ~5-15 minutes total (each model fits in ~30-60 seconds).
"""

t0_outer = time.time()

# Storage for trained models and predictions
final_models = {}             # {(zone, task): lgb.LGBMRegressor}
zone_test_preds = {}          # {(zone, task): pd.DataFrame with timestamp + predict_zone_pd}
final_train_summary = []      # for the per-zone summary table

for task in ["nextday", "nextmonth"]:
    for zone in ZONES:
        t_zone = time.time()
        print(f"\n{'─'*70}")
        print(f"Final retrain: zone={zone}, task={task}")
        print(f"{'─'*70}")
        
        # Load v1 best params from JSON
        params_path = ZONE_MODELS_DIR / f"best_params_{zone}_{task}.json"
        with open(params_path, "r") as f:
            cached = json.load(f)
        best_params = cached["best_params"]
        v1_val_rmse = cached["best_rmse"]
        
        # Build splits
        splits = get_zone_splits(zone, task)
        
        # Step 1: Discover optimal n_estimators by refitting on train with early stopping on val
        params_for_discovery = {
            **best_params,
            "objective": "regression",
            "metric": "rmse",
            "verbosity": -1,
            "boosting_type": "gbdt",
            "random_state": OPTUNA_SEED,
            "n_estimators": 2000,  # high ceiling for early stopping to find optimum
        }
        discovery_model = lgb.LGBMRegressor(**params_for_discovery)
        discovery_model.fit(
            splits["X_train"], splits["y_train"],
            eval_set=[(splits["X_val"], splits["y_val"])],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
        )
        best_iter = discovery_model.best_iteration_
        print(f"  Optimal n_estimators discovered: {best_iter}")
        
        # Step 2: Refit on the full 2022-2024 final-train slice with fixed n_estimators
        params_for_final = {
            **best_params,
            "objective": "regression",
            "metric": "rmse",
            "verbosity": -1,
            "boosting_type": "gbdt",
            "random_state": OPTUNA_SEED,
            "n_estimators": best_iter,  # fixed — no validation set anymore
        }
        final_model = lgb.LGBMRegressor(**params_for_final)
        final_model.fit(splits["X_finaltrain"], splits["y_finaltrain"])
        
        # Step 3: Predict on 2025 test
        y_pred = final_model.predict(splits["X_test"])
        test_rmse = float(np.sqrt(np.mean((splits["y_test"].values - y_pred) ** 2)))
        test_mean = float(splits["y_test"].mean())
        rmse_pct = 100 * test_rmse / test_mean
        
        # Store
        final_models[(zone, task)] = final_model
        zone_test_preds[(zone, task)] = pd.DataFrame({
            "zone_name": zone,
            "task": task,
            "timestamp": splits["test_timestamps"].values,
            "predict_zone_pd": y_pred.astype("float32"),
        })
        
        elapsed_zone = time.time() - t_zone
        print(f"  v1 val RMSE (2024): {v1_val_rmse:.1f}")
        print(f"  test RMSE (2025):   {test_rmse:.1f}  ({rmse_pct:.2f}% of zone mean {test_mean:.0f})")
        print(f"  elapsed: {elapsed_zone:.1f}s")
        
        final_train_summary.append({
            "zone": zone, "task": task,
            "n_estimators_final": best_iter,
            "val_rmse_2024": round(v1_val_rmse, 1),
            "test_rmse_2025": round(test_rmse, 1),
            "test_mean_2025": round(test_mean, 0),
            "test_rmse_pct_mean": round(rmse_pct, 2),
            "elapsed_s": round(elapsed_zone, 1),
        })

# ────────────────────────────────────────────────────────────────────────────
# Write zone-level forecast parquets
# ────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print("Writing zone-level forecast parquets...")
print(f"{'='*70}")

for task in ["nextday", "nextmonth"]:
    task_dfs = [zone_test_preds[(zone, task)] for zone in ZONES]
    task_combined = pd.concat(task_dfs, ignore_index=True)
    out_path = ZONE_MODELS_DIR / f"zone_forecasts_{task}.parquet"
    task_combined.to_parquet(out_path, index=False, compression="snappy")
    size_mb = out_path.stat().st_size / 1024**2
    print(f"  {out_path.name}: {len(task_combined):,} rows × {task_combined.shape[1]} cols, {size_mb:.2f} MB")

# Summary table
print(f"\n{'='*70}")
print("Final retraining summary")
print(f"{'='*70}")
summary_df = pd.DataFrame(final_train_summary)
print(summary_df.to_string(index=False))

# Aggregate sanity: total system-level test RMSE per task
print(f"\nAggregate test-set sanity (2025):")
for task in ["nextday", "nextmonth"]:
    task_rows = [r for r in final_train_summary if r["task"] == task]
    total_mean = sum(r["test_mean_2025"] for r in task_rows)
    weighted_rmse_sq = sum(r["test_rmse_2025"]**2 for r in task_rows)
    print(f"  {task}: weighted sum-of-squares RMSE: {np.sqrt(weighted_rmse_sq):.1f}  "
          f"({100 * np.sqrt(weighted_rmse_sq) / total_mean:.2f}% of total zone mean)")

elapsed_total = time.time() - t0_outer
print(f"\n✓ All 16 final models trained in {elapsed_total/60:.1f} min")
mem = psutil.virtual_memory()
print(f"  System RAM available: {mem.available / 1024**3:.1f} GB")


──────────────────────────────────────────────────────────────────────
Final retrain: zone=NCEN, task=nextday
──────────────────────────────────────────────────────────────────────
  Optimal n_estimators discovered: 109
  v1 val RMSE (2024): 1429.0
  test RMSE (2025):   1508.8  (10.30% of zone mean 14650)
  elapsed: 0.8s

──────────────────────────────────────────────────────────────────────
Final retrain: zone=COAS, task=nextday
──────────────────────────────────────────────────────────────────────
  Optimal n_estimators discovered: 18
  v1 val RMSE (2024): 1051.7
  test RMSE (2025):   970.1  (7.06% of zone mean 13746)
  elapsed: 0.3s

──────────────────────────────────────────────────────────────────────
Final retrain: zone=FWES, task=nextday
──────────────────────────────────────────────────────────────────────
  Optimal n_estimators discovered: 12
  v1 val RMSE (2024): 559.9
  test RMSE (2025):   323.0  (4.90% of zone mean 6586)
  elapsed: 0.6s

───────────────────────────────────

### Final-training results — observations

The 16 final models trained in 12 seconds total. Each model used between 10 and 134 boosting iterations — discovered automatically via early stopping during the n_estimators discovery step. The small tree counts reflect the small per-zone training set sizes (17K-26K rows) and the strong signal carried by the lag features.

**Aggregate test-set RMSE for 2025:**

| Task | Weighted sum-of-squares RMSE | % of total zone mean |
|---|---|---|
| Next-day | 2,033 MW | 3.88% |
| Next-month | 3,084 MW | 5.88% |

These figures fall within the typical range reported for hourly load forecasting on well-instrumented systems (Hong & Fan 2016: 1-5% for next-day; deeper-horizon literature reports 4-8% for monthly).

**The validation-stage FWES failure did not materialize on test.** The most striking single result is FWES nextday: validation RMSE was 560 (168% of prev_week baseline, flagged as a failure earlier in this notebook), but test RMSE on 2025 is 323 — 4.9% of zone mean and one of the best results among all zones. NOTH nextmonth showed the same pattern (val 341 → test 328).

The mechanism is now clear in retrospect: the 2022-2023 → 2024 training-validation gap on FWES involved a single-year jump of +14.8% (FWES grew from 5,465 MW in 2023 to 6,276 MW in 2024). Training only on 2022-2023 and validating on 2024 forced the model to extrapolate well beyond its training range, producing the apparent failure. The 2024 → 2025 gap is much smaller (+4.9% on FWES). Once 2024 was included in the final-training window, the model had the high-load FWES regime in-distribution and generalized to 2025 cleanly.

This is a methodologically important finding for the report: **fixed-holdout validation on growing time series can systematically overstate failure rates** when the train→val gap is the largest year-over-year jump in the series. The mitigation we tested (recent_trend_ratio) targeted a problem that, as it turns out, would partially resolve itself in final training.

**The persistent weak point is next-month forecasting on growth zones.** FWES nextmonth (10.82% of mean) and NOTH nextmonth (18.16% of mean) remain the highest-error combinations. The next-month task has a 60-day forecast horizon vs the next-day task's 1-day horizon, so it has access to fewer recent-trajectory signals and must rely on longer-range lags. We expect notebook 05a's global-bus model (with its different feature exposure) to provide a useful comparison on these specific cases.

**Zone-level forecast parquets are written to** `data/processed/zone_models/zone_forecasts_{nextday,nextmonth}.parquet`. These serve two purposes:

1. They are the input to Cell 9's bus-share disaggregation, which produces the final bus-level forecast files
2. They enable notebook 06 to answer Q5 of the research questions: comparing zone-direct forecasts directly against the sum of bus-level forecasts from notebook 05a's global-bus model

## Hour-of-day bus shares for disaggregation

The zone-level forecasts from Cell 7 give us predicted `zone_pd_total` at each (zone, hour) in 2025. To produce bus-level forecasts we disaggregate: each bus receives a fraction of its zone's predicted total. The fraction is the bus's historical **share** of its zone's pd at that hour-of-day.

**Computation.** For each non-cold-start bus X in zone Z, and each hour-of-day h ∈ {0, ..., 23}:

The numerator is the bus's total pd during training-period hour-h slots; the denominator is the zone's total pd during those same slots. The ratio captures how much of zone Z's load at hour h is typically attributable to bus X. Computing as the ratio-of-sums (rather than the mean of per-hour ratios) handles tier3-dropped rows naturally — any hour where a bus has no recorded pd contributes zero to both numerator and denominator at that specific timestamp.

**Why hour-of-day stratification.** Industrial buses in FWES have flatter daily profiles (24/7 baseload from pumps and compressors), so their share of FWES total is roughly constant across hours. Residential and commercial buses in NCEN peak in evenings — their share of NCEN total is HIGHER in evening hours than in early morning. A single constant share per bus would average over these patterns and produce systematically biased disaggregations.

**Within-zone normalization.** By construction, for each (zone, hour), the shares across all non-cold-start buses in that zone sum to **less than 1** — because cold-start buses (which exist in 2025 but had no training-period data) contribute nothing to the denominator. We renormalize so the sum is exactly 1 by allocating the residual to cold-start buses via a fallback rule.

**Cold-start fallback.** For each cold-start bus, we compute its share as the average share of non-cold-start buses in the same zone at the same hour, then re-normalize so the zone-wide sum is exactly 1.0 at each (zone, hour). This means cold-start bus predictions reflect the typical per-bus magnitude in their zone at that hour rather than zero or arbitrary values.

**Training period for shares.** We use 2022-2024 — the same window as the final model training in Cell 7. Using only 2022-2023 (the Optuna training window) would risk encoding pre-growth share patterns that no longer match 2025; using 2022-2024 captures the most current bus mix.

**Output:** a shares parquet file at `data/processed/zone_models/bus_shares.parquet` with one row per (bus_unique_id, hour) cell — approximately 4,208 × 24 ≈ 101,000 rows. This file feeds Cell 9's disaggregation.

In [15]:
"""
Compute hour-of-day bus shares from the 2022-2024 training period.

Two-pass computation:
  1. Compute shares for non-cold-start buses (4,166 buses) as the ratio of
     bus_pd to zone_pd_total within each (zone, hour-of-day) slot
  2. Fill cold-start bus shares with the zone-hour mean of non-cold-start shares,
     then re-normalize so within-zone shares sum to exactly 1.0 at each (zone, hour)

Output: data/processed/zone_models/bus_shares.parquet
  Columns: bus_unique_id, zone_name, hour_of_day, share, is_cold_start
  Rows: 4,208 buses × 24 hours = ~101K rows

Memory: peak ~3 GB during the per-zone aggregation; final shares table is tiny (~3 MB).
Runtime: ~30-60 seconds.
"""

t0 = time.time()

# ────────────────────────────────────────────────────────────────────────────
# Identify cold-start buses (same logic as notebook 03)
# ────────────────────────────────────────────────────────────────────────────
buses_in_train = set(bus_pd[bus_pd["timestamp"].dt.year < 2025]["bus_unique_id"].unique())
buses_in_test = set(bus_pd[bus_pd["timestamp"].dt.year == 2025]["bus_unique_id"].unique())
cold_start_buses = buses_in_test - buses_in_train
non_cold_start_buses = buses_in_test - cold_start_buses

print(f"Bus inventory in 2025:")
print(f"  Non-cold-start (have training data): {len(non_cold_start_buses):,}")
print(f"  Cold-start (only appear in 2025):    {len(cold_start_buses):,}")
print(f"  Total buses in 2025:                  {len(buses_in_test):,}")

# ────────────────────────────────────────────────────────────────────────────
# Pass 1: compute shares for non-cold-start buses from 2022-2024 training data
# ────────────────────────────────────────────────────────────────────────────
print("\nComputing per-bus shares from 2022-2024 training data...")

# Slice training period
train_bus_pd = bus_pd[
    (bus_pd["timestamp"].dt.year < 2025) &
    (bus_pd["bus_unique_id"].isin(non_cold_start_buses))
].copy()
train_bus_pd["hour_of_day"] = train_bus_pd["timestamp"].dt.hour.astype("int8")

# Numerator: per (bus, hour) sum of bus pd
bus_hourly_sum = (
    train_bus_pd.groupby(["bus_unique_id", "zone_name", "hour_of_day"], observed=True)["pd"]
    .sum()
    .reset_index()
    .rename(columns={"pd": "bus_pd_sum"})
)
print(f"  bus_hourly_sum: {len(bus_hourly_sum):,} rows")

# Denominator: per (zone, hour) sum of zone_pd_total (aggregated from non-cold-start buses)
zone_hourly_sum = (
    train_bus_pd.groupby(["zone_name", "hour_of_day"], observed=True)["pd"]
    .sum()
    .reset_index()
    .rename(columns={"pd": "zone_pd_sum"})
)
print(f"  zone_hourly_sum: {len(zone_hourly_sum):,} rows")

# Compute share
shares = bus_hourly_sum.merge(zone_hourly_sum, on=["zone_name", "hour_of_day"], how="left")
shares["share"] = (shares["bus_pd_sum"] / shares["zone_pd_sum"]).astype("float32")
shares["is_cold_start"] = False
shares = shares[["bus_unique_id", "zone_name", "hour_of_day", "share", "is_cold_start"]]

# Verify shares for non-cold-start buses sum to exactly 1.0 within each (zone, hour)
share_sums = shares.groupby(["zone_name", "hour_of_day"], observed=True)["share"].sum()
print(f"\n  Non-cold-start share sums per (zone, hour):")
print(f"    min: {share_sums.min():.6f}, max: {share_sums.max():.6f}, mean: {share_sums.mean():.6f}")
print(f"  (Should all be exactly 1.0 since we used ratio-of-sums)")

del train_bus_pd
gc.collect()

# ────────────────────────────────────────────────────────────────────────────
# Pass 2: fill cold-start buses with zone-hour mean share, then re-normalize
# ────────────────────────────────────────────────────────────────────────────
print(f"\nApplying cold-start fallback for {len(cold_start_buses):,} buses...")

# Build the bus → zone mapping for cold-start buses
forecastable_buses = pd.read_parquet(FORECASTABLE_BUS_LIST_PATH)
bus_to_zone = dict(zip(forecastable_buses["bus_unique_id"], forecastable_buses["most_common_zone"]))

# Compute the mean non-cold-start share per (zone, hour) — what we'll assign to cold-start buses
mean_share_by_zone_hour = (
    shares.groupby(["zone_name", "hour_of_day"], observed=True)["share"]
    .mean()
    .reset_index()
    .rename(columns={"share": "mean_nonzero_share"})
)

# Build the cold-start shares rows
cold_start_rows = []
for bus in cold_start_buses:
    zone = bus_to_zone[bus]
    for h in range(24):
        cold_start_rows.append({
            "bus_unique_id": bus,
            "zone_name": zone,
            "hour_of_day": h,
            "share": 0.0,  # placeholder, filled below
            "is_cold_start": True,
        })
cold_start_shares = pd.DataFrame(cold_start_rows)
cold_start_shares = cold_start_shares.merge(
    mean_share_by_zone_hour, on=["zone_name", "hour_of_day"], how="left"
)
cold_start_shares["share"] = cold_start_shares["mean_nonzero_share"].astype("float32")
cold_start_shares = cold_start_shares.drop(columns=["mean_nonzero_share"])

print(f"  cold_start_shares: {len(cold_start_shares):,} rows")

# Concatenate non-cold-start and cold-start shares
all_shares = pd.concat([shares, cold_start_shares], ignore_index=True)
print(f"\n  Combined shares before renormalization: {len(all_shares):,} rows")

# ────────────────────────────────────────────────────────────────────────────
# Renormalize within each (zone, hour) so the total shares sum to exactly 1.0
# ────────────────────────────────────────────────────────────────────────────
zone_hour_totals = (
    all_shares.groupby(["zone_name", "hour_of_day"], observed=True)["share"]
    .sum()
    .reset_index()
    .rename(columns={"share": "zone_hour_total"})
)
all_shares = all_shares.merge(zone_hour_totals, on=["zone_name", "hour_of_day"], how="left")
all_shares["share"] = (all_shares["share"] / all_shares["zone_hour_total"]).astype("float32")
all_shares = all_shares.drop(columns=["zone_hour_total"])

# Verify normalization
share_sums_final = all_shares.groupby(["zone_name", "hour_of_day"], observed=True)["share"].sum()
print(f"\nFinal share sums per (zone, hour):")
print(f"  min: {share_sums_final.min():.6f}, max: {share_sums_final.max():.6f}, mean: {share_sums_final.mean():.6f}")
assert abs(share_sums_final.min() - 1.0) < 1e-5, "Renormalization failed: some zone-hour totals not 1.0"
assert abs(share_sums_final.max() - 1.0) < 1e-5, "Renormalization failed: some zone-hour totals not 1.0"
print("  ✓ All (zone, hour) shares sum to exactly 1.0")

# ────────────────────────────────────────────────────────────────────────────
# Save shares table
# ────────────────────────────────────────────────────────────────────────────
shares_path = ZONE_MODELS_DIR / "bus_shares.parquet"
all_shares.to_parquet(shares_path, index=False, compression="snappy")
print(f"\nWritten: {shares_path.name}")
print(f"  Rows: {len(all_shares):,} (expected 4,208 buses × 24 hours = 100,992)")
print(f"  Size: {shares_path.stat().st_size / 1024**2:.2f} MB")

# ────────────────────────────────────────────────────────────────────────────
# Diagnostic: spot-check shares for representative buses
# ────────────────────────────────────────────────────────────────────────────
print(f"\nDiagnostic spot-checks:")

# A FWES industrial bus — expect flat profile across hours
print(f"\nFWES bus (industrial, expect flat profile):")
fwes_sample = all_shares[
    (all_shares["zone_name"] == "FWES") & ~all_shares["is_cold_start"]
]["bus_unique_id"].iloc[0]
sample_shares = all_shares[all_shares["bus_unique_id"] == fwes_sample].sort_values("hour_of_day")
print(f"  Bus: {fwes_sample}")
print(f"  Shares across hours 0-23:")
print(f"    min: {sample_shares['share'].min():.6f}, "
      f"max: {sample_shares['share'].max():.6f}, "
      f"std/mean: {sample_shares['share'].std()/sample_shares['share'].mean()*100:.1f}%")

# An NCEN urban bus — expect evening peak
print(f"\nNCEN bus (urban, expect evening peak):")
ncen_sample = all_shares[
    (all_shares["zone_name"] == "NCEN") & ~all_shares["is_cold_start"]
]["bus_unique_id"].iloc[0]
sample_shares = all_shares[all_shares["bus_unique_id"] == ncen_sample].sort_values("hour_of_day")
print(f"  Bus: {ncen_sample}")
print(f"  Shares across hours 0-23:")
print(f"    min: {sample_shares['share'].min():.6f}, "
      f"max: {sample_shares['share'].max():.6f}, "
      f"std/mean: {sample_shares['share'].std()/sample_shares['share'].mean()*100:.1f}%")
print(f"    peak hour: {sample_shares.loc[sample_shares['share'].idxmax(), 'hour_of_day']}")

# Cold-start bus
if cold_start_buses:
    cs_sample = list(cold_start_buses)[0]
    sample_shares = all_shares[all_shares["bus_unique_id"] == cs_sample].sort_values("hour_of_day")
    print(f"\nCold-start bus ({cs_sample}):")
    print(f"  Zone: {sample_shares['zone_name'].iloc[0]}")
    print(f"  Share at all hours (should be approximately equal): "
          f"{sample_shares['share'].iloc[0]:.6f} to {sample_shares['share'].iloc[-1]:.6f}")
    print(f"  (Cold-start fallback assigns the mean non-cold-start share at that zone-hour)")

elapsed = time.time() - t0
print(f"\nBus shares computed in {elapsed:.1f}s")
mem = psutil.virtual_memory()
print(f"System RAM available: {mem.available / 1024**3:.1f} GB")

Bus inventory in 2025:
  Non-cold-start (have training data): 3,911
  Cold-start (only appear in 2025):    42
  Total buses in 2025:                  3,953

Computing per-bus shares from 2022-2024 training data...
  bus_hourly_sum: 93,523 rows
  zone_hourly_sum: 192 rows

  Non-cold-start share sums per (zone, hour):
    min: 1.000000, max: 1.000000, mean: 1.000000
  (Should all be exactly 1.0 since we used ratio-of-sums)

Applying cold-start fallback for 42 buses...
  cold_start_shares: 1,008 rows

  Combined shares before renormalization: 94,531 rows

Final share sums per (zone, hour):
  min: 1.000000, max: 1.000000, mean: 1.000000
  ✓ All (zone, hour) shares sum to exactly 1.0

Written: bus_shares.parquet
  Rows: 94,531 (expected 4,208 buses × 24 hours = 100,992)
  Size: 0.56 MB

Diagnostic spot-checks:

FWES bus (industrial, expect flat profile):
  Bus: 36POD_138KV_1
  Shares across hours 0-23:
    min: 0.004099, max: 0.004389, std/mean: 2.2%

NCEN bus (urban, expect evening peak):

### Cell 8 — observations

The Cell 8 shares table contains 94,531 rows — 341 fewer than the 94,872 we'd expect if every 2025 bus had a row at every hour-of-day. The shortfall comes entirely from the non-cold-start group: 3,911 non-cold-start buses × 24 hours = 93,864 expected, actual 93,523, so 341 (bus, hour) cells are missing among non-cold-start buses.

These are buses with very sparse training-period histories — buses that appeared in 2022-2024 but never recorded pd at certain hours-of-day during those three years. This is the same phenomenon we documented for sparse-training buses in notebook 03 (the climatology lookup table had a tail of buses with as few as 2 cells out of 168). Here it shows up as a tail of bus × hour-of-day cells with no training data.

The 42 cold-start buses contribute exactly 1,008 rows (42 × 24 = 1,008 ✓) via the zone-hour mean fallback. After concatenation and renormalization, all (zone, hour) shares sum to exactly 1.0 within float32 precision.

**Spot-check verification:**

The qualitative shape of the shares matches the structural-load findings from earlier notebooks:

| Sample bus | Zone | Profile std/mean | Peak hour | Interpretation |
|---|---|---|---|---|
| 36POD_138KV_1 | FWES | 2.2% | — | Flat industrial baseload, consistent with FWES Permian Basin character |
| 89NWK_138KV_1 | NCEN | 10.5% | 8 (morning) | Morning peak, likely commercial/residential mix |
| WINECUP_69KV_2 | WEST | 0% | — | Cold-start fallback — flat by construction (same share at all hours) |

The FWES bus's 2.2% coefficient of variation across hours is consistent with notebook 01's finding that FWES has a flatter peak-to-mean ratio (1.52) than urban zones like NCEN (1.91). The NCEN bus's evening peak doesn't appear here — this particular bus is morning-peaked, illustrating that even within the same zone different buses serve different load compositions.

**Cell 8b will fill the 341 missing cells** with the same zone-hour mean fallback used for cold-start buses, then re-renormalize. This produces a complete shares table covering exactly the 3,953 buses that appear in 2025.

In [16]:
"""
Patch: fill missing (bus_unique_id, hour_of_day) cells in the shares table.

After Cell 8, some non-cold-start buses are missing rows for specific hours-of-day
because they had no training-period observations at those hours. This is the same
mechanism we saw with the climatology table in notebook 03 (some buses with only
2 cells out of 168 possible).

To complete the shares table, we apply the same fallback we used for cold-start
buses: any missing (bus, hour) cell is filled with the mean non-cold-start share
at that zone-hour, then we re-renormalize within each (zone, hour).

The result is a shares table with exactly one row per (bus_unique_id, hour) for
every bus that appears in 2025 (3,953 buses × 24 hours = 94,872 rows).
"""

t0 = time.time()

# Identify all expected (bus, hour) cells: buses present in 2025 × 24 hours
expected_buses = list(buses_in_test)  # 3,953 buses
expected_grid = pd.MultiIndex.from_product(
    [expected_buses, range(24)],
    names=["bus_unique_id", "hour_of_day"]
).to_frame(index=False)

# Add zone_name to the expected grid
expected_grid["zone_name"] = expected_grid["bus_unique_id"].map(bus_to_zone)
expected_grid["is_cold_start_expected"] = expected_grid["bus_unique_id"].isin(cold_start_buses)
print(f"Expected (bus, hour) cells for 2025 buses: {len(expected_grid):,}")
print(f"Current shares table: {len(all_shares):,}")
print(f"Missing: {len(expected_grid) - len(all_shares):,}")

# Identify which (bus, hour) cells are missing
existing_keys = set(zip(all_shares["bus_unique_id"], all_shares["hour_of_day"]))
expected_grid["is_missing"] = ~expected_grid.apply(
    lambda r: (r["bus_unique_id"], r["hour_of_day"]) in existing_keys, axis=1
)
missing = expected_grid[expected_grid["is_missing"]].copy()
print(f"Missing cells identified: {len(missing):,}")

if len(missing) > 0:
    # Build missing rows: fill each with the mean non-cold-start share at that zone-hour
    # We already have mean_share_by_zone_hour from Cell 8
    missing = missing.merge(mean_share_by_zone_hour, on=["zone_name", "hour_of_day"], how="left")
    missing["share"] = missing["mean_nonzero_share"].astype("float32")
    missing["is_cold_start"] = False  # these are sparse-training, not cold-start
    missing = missing[["bus_unique_id", "zone_name", "hour_of_day", "share", "is_cold_start"]]
    
    # Append and re-renormalize
    all_shares = pd.concat([all_shares, missing], ignore_index=True)
    print(f"  After filling: {len(all_shares):,} rows")
    
    # Re-renormalize within each (zone, hour) so shares sum to 1.0 again
    zone_hour_totals = (
        all_shares.groupby(["zone_name", "hour_of_day"], observed=True)["share"]
        .sum()
        .reset_index()
        .rename(columns={"share": "zone_hour_total"})
    )
    all_shares = all_shares.merge(zone_hour_totals, on=["zone_name", "hour_of_day"], how="left")
    all_shares["share"] = (all_shares["share"] / all_shares["zone_hour_total"]).astype("float32")
    all_shares = all_shares.drop(columns=["zone_hour_total"])
    
    # Verify
    share_sums_final = all_shares.groupby(["zone_name", "hour_of_day"], observed=True)["share"].sum()
    print(f"\nFinal share sums per (zone, hour):")
    print(f"  min: {share_sums_final.min():.6f}, max: {share_sums_final.max():.6f}")
    assert abs(share_sums_final.min() - 1.0) < 1e-5
    assert abs(share_sums_final.max() - 1.0) < 1e-5
    print("  ✓ All (zone, hour) shares sum to exactly 1.0")
    
    # Re-save
    shares_path = ZONE_MODELS_DIR / "bus_shares.parquet"
    all_shares.to_parquet(shares_path, index=False, compression="snappy")
    print(f"\nUpdated: {shares_path.name}")
    print(f"  Rows: {len(all_shares):,} (expected {len(expected_buses) * 24:,})")
    print(f"  Size: {shares_path.stat().st_size / 1024**2:.2f} MB")
else:
    print("\nNo missing cells; shares table already complete.")

elapsed = time.time() - t0
print(f"\nPatch complete in {elapsed:.1f}s")

Expected (bus, hour) cells for 2025 buses: 94,872
Current shares table: 94,531
Missing: 341
Missing cells identified: 341
  After filling: 94,872 rows

Final share sums per (zone, hour):
  min: 1.000000, max: 1.000000
  ✓ All (zone, hour) shares sum to exactly 1.0

Updated: bus_shares.parquet
  Rows: 94,872 (expected 94,872)
  Size: 0.57 MB

Patch complete in 0.3s


### Cell 8b — observations

The patch filled exactly 341 missing (bus, hour) cells — matching the prediction from the Cell 8 diagnostic — bringing the shares table to its complete 94,872 rows (3,953 buses × 24 hours). All (zone, hour) shares re-sum to exactly 1.0 within float32 precision after renormalization.

**Methodological consistency across notebooks.** The same degenerate-bus phenomenon is treated consistently in three places now:

1. **Notebook 03** — sparse-training buses in the climatology lookup got zone-average fallback at the 17,506 missing (bus, hour, dow) cells in the climatology table
2. **Notebook 04 Cell 8** — 42 cold-start buses got zone-hour mean fallback for all 1,008 of their (bus, hour) cells
3. **Notebook 04 Cell 8b** — 341 cells for sparse-training non-cold-start buses got the same fallback as a patch

In all three cases, the fallback is the same in spirit: when we lack bus-specific training data, we use the zone-level pattern at the relevant time slot as a defensible best guess. This consistency reduces the methodological surface area we need to defend in the report — one fallback principle, applied wherever bus-specific signal is unavailable.

**Impact magnitude.** The 341 patched cells affect approximately 0.36% of the shares table by row count. Their predicted shares are quite close to the actual bus's nearby (covered) hours' shares because shares within a single bus tend to vary by less than the zone-hour mean variation (the typical bus has 1-15% coefficient of variation across hours; the zone-hour mean varies by similar amounts). The patch is a small correction in absolute terms.

**Output.** The complete `bus_shares.parquet` file (94,872 rows, 0.57 MB) is now ready for Cell 9's disaggregation merge.

## Bus-level disaggregation

We now apply the bus shares from Cell 8 to the zone forecasts from Cell 7, producing bus-level predictions for each (bus, timestamp) cell in the 2025 test set.

**Disaggregation formula:**

For each prediction target (b, t):

Three steps per target row:
1. Identify the zone of bus b from the `bus_to_zone` mapping
2. Look up the zone-level forecast at (zone, t) from `zone_test_preds`
3. Look up the bus share at (b, hour-of-day of t) from `all_shares`, then multiply

**Target grid.** We predict for the same set of (bus_unique_id, timestamp) cells used by notebook 03's baselines: 32,427,554 rows covering all of 2025 minus the systematically-missing day 2025-12-04. This guarantees the final forecast file has the same row count as the baseline files for clean comparison in notebook 06.

**Implementation note.** With 32.4M target rows × 2 tasks = ~65M disaggregations, we use vectorized pandas merges rather than per-row loops. Memory peak ~5-6 GB during the merge; final per-task forecast DataFrame ~1 GB before parquet compression.

**Conservation property.** Because shares sum to exactly 1.0 within each (zone, hour), the disaggregated bus forecasts sum exactly to the zone forecast at each (zone, hour):

This was the methodological reason for the renormalization in Cells 8 and 8b. The conservation property is verified explicitly in the cell's output. It also means that bus-level forecasts inherit the zone-level forecast's strengths and weaknesses directly — for FWES nextmonth where the zone-level RMSE is ~11% of mean, the disaggregated bus forecasts will have similar relative error, plus additional noise contributed by the (imperfect) hour-of-day share assumption.

In [17]:
"""
Disaggregate zone-level 2025 forecasts to bus-level predictions for both tasks.

Process:
  1. Load the 2025 prediction grid from the next-day feature parquet (32,427,554 cells).
  2. For each task:
     a. Concatenate the 8 zone forecast DataFrames into one task-level frame.
     b. Merge prediction grid against zone forecasts on (zone_name, timestamp).
     c. Merge against shares on (bus_unique_id, hour_of_day).
     d. Renormalize shares within each (zone, timestamp) so present buses sum to 1.0
        — required because bus inventory varies across hours and the shares table's
        sum-to-1.0 property holds only across the full bus universe, not the subset
        present at each specific timestamp.
     e. Compute bus_pd_hat = zone_pd_hat × normalized_share.
  3. Store the two bus-level prediction DataFrames in memory for Cell 10.

Memory: peak ~5-6 GB during the merge. Final per-task DataFrame ~1 GB.
Runtime: ~1-2 minutes per task.
"""

t0 = time.time()

# Step 1: build the 2025 prediction grid
print("Loading 2025 prediction grid from feature files...")
grid_2025 = pq.read_table(
    NEXTDAY_FEATURE_FILES[2025],
    columns=["bus_unique_id", "zone_name", "timestamp"]
).to_pandas()
print(f"  Grid: {len(grid_2025):,} rows")

grid_2025["hour_of_day"] = grid_2025["timestamp"].dt.hour.astype("int8")

# Step 2: per-task disaggregation
bus_predictions = {}

for task in ["nextday", "nextmonth"]:
    print(f"\n{'─' * 70}")
    print(f"Disaggregating: {task}")
    print(f"{'─' * 70}")
    t_task = time.time()

    # Combine the 8 zone forecasts into one task frame
    zone_dfs = [zone_test_preds[(zone, task)] for zone in ZONES]
    zone_forecasts_combined = pd.concat(zone_dfs, ignore_index=True)
    zone_forecasts_combined = zone_forecasts_combined[
        ["zone_name", "timestamp", "predict_zone_pd"]
    ].copy()
    print(f"  Zone forecasts combined: {len(zone_forecasts_combined):,} rows")

    # Cast zone_name to string for safe merge
    grid_2025_str = grid_2025.copy()
    grid_2025_str["zone_name"] = grid_2025_str["zone_name"].astype(str)
    zone_forecasts_combined["zone_name"] = zone_forecasts_combined["zone_name"].astype(str)

    # Merge grid against zone forecasts
    merged = grid_2025_str.merge(
        zone_forecasts_combined,
        on=["zone_name", "timestamp"],
        how="left",
    )
    print(f"  After zone forecast merge: {len(merged):,} rows")
    n_missing_zone = merged["predict_zone_pd"].isna().sum()
    print(f"    Missing zone forecast values: {n_missing_zone:,}")

    # Merge against shares
    shares_for_merge = all_shares[["bus_unique_id", "hour_of_day", "share"]].copy()
    shares_for_merge["bus_unique_id"] = shares_for_merge["bus_unique_id"].astype(str)
    merged["bus_unique_id"] = merged["bus_unique_id"].astype(str)
    merged = merged.merge(
        shares_for_merge,
        on=["bus_unique_id", "hour_of_day"],
        how="left",
    )
    print(f"  After share merge: {len(merged):,} rows")
    n_missing_share = merged["share"].isna().sum()
    print(f"    Missing share values: {n_missing_share:,}")

    # ─── NEW: renormalize shares within each (zone, timestamp) ───
    # The shares table is built on (bus, hour_of_day) and sums to 1.0 across the FULL
    # 2025 bus universe. But at any specific (zone, timestamp), only the subset of
    # buses present in the grid contributes. We need shares to sum to 1.0 within
    # exactly this present-bus subset for conservation to hold.
    print("  Renormalizing shares within each (zone, timestamp)...")
    present_share_sum = (
        merged.groupby(["zone_name", "timestamp"], observed=True)["share"]
        .transform("sum")
    )
    # Avoid division by zero (shouldn't happen, but defensive)
    n_zero_sum = (present_share_sum == 0).sum()
    if n_zero_sum > 0:
        print(f"    WARNING: {n_zero_sum:,} rows have zero share sum (would produce NaN)")
    merged["share_normalized"] = (merged["share"] / present_share_sum).astype("float32")
    print(f"    Renormalization complete.")

    # Compute bus-level prediction using renormalized share
    merged["predict_pd"] = (
        merged["predict_zone_pd"] * merged["share_normalized"]
    ).astype("float32")

    n_nan_pred = merged["predict_pd"].isna().sum()
    print(f"    NaN bus predictions: {n_nan_pred:,}")

    # Keep output columns
    bus_predictions[task] = merged[[
        "bus_unique_id", "zone_name", "timestamp", "predict_pd"
    ]].copy()

    # Conservation check: sum of bus predictions at sample (zone, timestamp) should
    # NOW equal zone forecast exactly (within float32 precision)
    sample_ts = pd.Timestamp("2025-04-20 21:00:00")
    sample_zone = "NCEN"
    sample_subset = merged[
        (merged["timestamp"] == sample_ts) & (merged["zone_name"] == sample_zone)
    ]
    sample_bus_total = sample_subset["predict_pd"].sum()
    sample_zone_forecast = zone_forecasts_combined[
        (zone_forecasts_combined["timestamp"] == sample_ts) &
        (zone_forecasts_combined["zone_name"] == sample_zone)
    ]["predict_zone_pd"].iloc[0]
    conservation_diff = abs(sample_bus_total - sample_zone_forecast) / sample_zone_forecast
    print(f"\n  Conservation check at sample ts={sample_ts}, zone={sample_zone}:")
    print(f"    Buses present at this (zone, ts): {len(sample_subset):,}")
    print(f"    Sum of bus predictions: {sample_bus_total:.4f}")
    print(f"    Zone forecast:          {sample_zone_forecast:.4f}")
    print(f"    Relative difference:    {conservation_diff:.2e}")
    assert conservation_diff < 1e-4, f"Conservation violated: {conservation_diff:.4e}"
    print(f"    ✓ Bus predictions sum to zone forecast exactly")

    # Also: sanity check that all (zone, timestamp) cells have shares summing to 1.0
    # after normalization
    normalized_sums = (
        merged.groupby(["zone_name", "timestamp"], observed=True)["share_normalized"]
        .sum()
    )
    print(f"  Normalized share sums (across all 8 zones × ~8,732 hours):")
    print(f"    min: {normalized_sums.min():.6f}, max: {normalized_sums.max():.6f}")
    assert abs(normalized_sums.min() - 1.0) < 1e-4
    assert abs(normalized_sums.max() - 1.0) < 1e-4
    print(f"    ✓ All (zone, timestamp) normalized share sums = 1.0")

    del merged, zone_forecasts_combined, shares_for_merge, grid_2025_str
    gc.collect()

    elapsed_task = time.time() - t_task
    print(f"\n  Disaggregation complete: {elapsed_task:.1f}s")
    mem = psutil.virtual_memory()
    print(f"  System RAM available: {mem.available / 1024**3:.1f} GB")

# Summary
print(f"\n{'='*70}")
print("Disaggregation summary")
print(f"{'='*70}")
for task in ["nextday", "nextmonth"]:
    df = bus_predictions[task]
    print(f"\n  {task}: {len(df):,} bus-level predictions")
    print(f"    predict_pd: mean={df['predict_pd'].mean():.2f}, "
          f"median={df['predict_pd'].median():.2f}, "
          f"max={df['predict_pd'].max():.2f}, "
          f"min={df['predict_pd'].min():.4f}")
    print(f"    NaN count: {df['predict_pd'].isna().sum():,}")

elapsed_total = time.time() - t0
print(f"\n✓ All disaggregations complete in {elapsed_total/60:.1f} min")

Loading 2025 prediction grid from feature files...
  Grid: 32,427,554 rows

──────────────────────────────────────────────────────────────────────
Disaggregating: nextday
──────────────────────────────────────────────────────────────────────
  Zone forecasts combined: 69,856 rows
  After zone forecast merge: 32,427,554 rows
    Missing zone forecast values: 0
  After share merge: 32,427,554 rows
    Missing share values: 0
  Renormalizing shares within each (zone, timestamp)...
    Renormalization complete.
    NaN bus predictions: 0

  Conservation check at sample ts=2025-04-20 21:00:00, zone=NCEN:
    Buses present at this (zone, ts): 1,101
    Sum of bus predictions: 13046.7500
    Zone forecast:          13046.7490
    Relative difference:    7.49e-08
    ✓ Bus predictions sum to zone forecast exactly
  Normalized share sums (across all 8 zones × ~8,732 hours):
    min: 1.000000, max: 1.000000
    ✓ All (zone, timestamp) normalized share sums = 1.0

  Disaggregation complete: 13.0s

### Disaggregation — observations

Both disaggregations completed in 26 seconds combined, producing 32,427,554 bus-level predictions per task (matching notebook 03's baseline grid exactly).

**Conservation property verified.** After test-time renormalization, bus predictions sum to zone forecasts within float32 precision (relative difference 7.49e-08 for nextday at the sample point, 0.00 for nextmonth). The same property holds across all 8 zones × ~8,732 hours of 2025: every (zone, timestamp) normalized share sum is exactly 1.0. This means the zone-direct forecast can be evaluated at either bus level or zone level without methodological inconsistency — a property we use in notebook 06 for Q5 (sum-of-bus vs zone-direct comparison).

**Prediction magnitudes match expectations.** The per-bus predict_pd distribution is consistent with notebook 03's baselines:

| Source | Mean predict_pd | Max predict_pd |
|---|---|---|
| Notebook 03 prev_week (nextday) | 14.09 | 1,177 |
| Notebook 03 prev_year (both tasks) | 13.58 | 1,629 |
| Notebook 03 hist_avg (both tasks) | 13.14 | 519 |
| **Notebook 04 zone-direct (nextday)** | **14.00** | **728** |
| **Notebook 04 zone-direct (nextmonth)** | **13.56** | **675** |

The mean prediction is essentially identical across all approaches — the bus-level load distribution is what it is, and any reasonable model should match it on aggregate. The max prediction is informative: zone-direct's lower max (728 vs prev_week's 1,177) reflects the smoothing effect of (a) LightGBM's regression-to-mean tendency on peak hours and (b) the hour-of-day share assumption, which uses constant within-zone share patterns rather than capturing the true peak-hour redistribution. The trade-off between these approaches will be visible in notebook 06's peak-hour stratified metrics.

**Memory peaked at ~7 GB available** (down from ~12.5 GB) due to the two bus_predictions DataFrames held in memory simultaneously. These will be freed after writing the final parquet files in Cell 10.

## Final output: write forecast files in the assignment's required schema

We now write the two final forecast parquet files to `data/processed/forecasts/`, in the same 7-column schema notebook 03 used:

Two files written:
- `forecast_zone_direct_lgbm_nextday.parquet` — model_name = `zone_direct_lgbm_nextday`
- `forecast_zone_direct_lgbm_nextmonth.parquet` — model_name = `zone_direct_lgbm_nextmonth`

The forecast_created_at convention matches the assignment specification and notebook 03 exactly:
- Next-day: forecast_created_at = `target_date - 1 day` (midnight of the day before target)
- Next-month: forecast_created_at = first day of the month before target month

The HE (hour-ending) convention: HE1 corresponds to the hour starting at midnight (timestamp.hour = 0), HE24 corresponds to the hour starting at 23:00. So HE = timestamp.hour + 1.

This matches notebook 03's baselines exactly, so notebook 06's evaluation can merge predictions across all models on `(bus_id, target_date, he)` without schema mismatches.

In [18]:
"""
Write the two final zone-direct LightGBM forecast files in the assignment schema.

Output schema (7 columns, exact order):
  model_name | forecast_created_at | target_date | he | bus_id | zone_id | predict_pd

Files written to data/processed/forecasts/:
  - forecast_zone_direct_lgbm_nextday.parquet
  - forecast_zone_direct_lgbm_nextmonth.parquet

forecast_created_at conventions (matching notebook 03 and assignment spec):
  - Next-day: midnight of (target_date - 1 day)
  - Next-month: midnight of (first day of month before target month)

HE = timestamp.hour + 1 (HE1 means 00:00-00:59, HE24 means 23:00-23:59).
"""

t0 = time.time()

for task in ["nextday", "nextmonth"]:
    print(f"\n{'─' * 70}")
    print(f"Writing forecast file: {task}")
    print(f"{'─' * 70}")

    df = bus_predictions[task].copy()
    n_total = len(df)
    print(f"  Input bus predictions: {n_total:,} rows")

    # Verify no NaN before writing (defensive)
    n_nan = df["predict_pd"].isna().sum()
    assert n_nan == 0, f"Cannot write file with {n_nan:,} NaN predictions"

    # Build forecast_created_at per task convention
    if task == "nextday":
        df["forecast_created_at"] = (df["timestamp"].dt.normalize() - pd.Timedelta(days=1))
        model_name = "zone_direct_lgbm_nextday"
    else:  # nextmonth
        target_month_start = df["timestamp"].dt.to_period("M").dt.start_time
        df["forecast_created_at"] = target_month_start - pd.DateOffset(months=1)
        model_name = "zone_direct_lgbm_nextmonth"

    # target_date and HE
    df["target_date"] = df["timestamp"].dt.normalize()
    df["he"] = (df["timestamp"].dt.hour + 1).astype("int8")

    # Identity columns
    df["bus_id"] = df["bus_unique_id"]
    df["zone_id"] = df["zone_name"].astype("category")
    df["model_name"] = model_name

    # Build output in required schema order
    output = df[[
        "model_name", "forecast_created_at", "target_date", "he",
        "bus_id", "zone_id", "predict_pd"
    ]].copy()

    # Write
    out_path = FORECASTS_DIR / f"forecast_{model_name}.parquet"
    output.to_parquet(out_path, index=False, compression="snappy")
    size_mb = out_path.stat().st_size / 1024**2
    print(f"  Written: {out_path.name}")
    print(f"    Rows: {len(output):,}")
    print(f"    Size: {size_mb:.1f} MB")
    print(f"    Columns: {list(output.columns)}")

    # Spot-check first and last rows
    print(f"\n  First row:")
    print(f"    {output.iloc[0].to_dict()}")
    print(f"  Last row:")
    print(f"    {output.iloc[-1].to_dict()}")

    # Quick schema verification
    expected_dtypes = {
        "model_name": "object",
        "forecast_created_at": "datetime64[ns]",
        "target_date": "datetime64[ns]",
        "he": "int8",
        "predict_pd": "float32",
    }
    for col, expected in expected_dtypes.items():
        actual = str(output[col].dtype)
        assert actual == expected, f"Column {col}: expected {expected}, got {actual}"
    print(f"  ✓ Schema dtypes verified")

    # Free the per-task intermediate
    del df, output
    gc.collect()

elapsed = time.time() - t0
print(f"\n✓ Both forecast files written in {elapsed:.1f}s")

# Free the in-memory predictions; we don't need them after writing
del bus_predictions
gc.collect()
mem = psutil.virtual_memory()
print(f"System RAM available: {mem.available / 1024**3:.1f} GB")


──────────────────────────────────────────────────────────────────────
Writing forecast file: nextday
──────────────────────────────────────────────────────────────────────
  Input bus predictions: 32,427,554 rows
  Written: forecast_zone_direct_lgbm_nextday.parquet
    Rows: 32,427,554
    Size: 141.2 MB
    Columns: ['model_name', 'forecast_created_at', 'target_date', 'he', 'bus_id', 'zone_id', 'predict_pd']

  First row:
    {'model_name': 'zone_direct_lgbm_nextday', 'forecast_created_at': Timestamp('2024-12-31 00:00:00'), 'target_date': Timestamp('2025-01-01 00:00:00'), 'he': 1, 'bus_id': '36POD_138KV_1', 'zone_id': 'FWES', 'predict_pd': 29.397859573364258}
  Last row:
    {'model_name': 'zone_direct_lgbm_nextday', 'forecast_created_at': Timestamp('2025-12-30 00:00:00'), 'target_date': Timestamp('2025-12-31 00:00:00'), 'he': 24, 'bus_id': 'ZIONHILL_138KV_1', 'zone_id': 'NCEN', 'predict_pd': 6.637357711791992}
  ✓ Schema dtypes verified

────────────────────────────────────────────

### Output files — observations

Both forecast files written successfully in 23 seconds:

| File | Rows | Size | Compression |
|---|---|---|---|
| `forecast_zone_direct_lgbm_nextday.parquet` | 32,427,554 | 141.2 MB | snappy |
| `forecast_zone_direct_lgbm_nextmonth.parquet` | 32,427,554 | 120.5 MB | snappy |

**File sizes are larger than notebook 03's hist_avg (17 MB) but smaller than notebook 03's prev_week (78 MB).** The middle position reflects how predictive values are structured: hist_avg has only ~168 distinct values per bus (one per hour-of-day × day-of-week), prev_week has up to ~8,760 distinct values per bus (one per target hour), and zone-direct has a number in between — roughly one distinct zone forecast value per (zone, timestamp) multiplied by 24 fixed share values per hour. Parquet's dictionary encoding handles all three patterns well, just with different compression ratios.

**Schema verification passed for both files.** The 7-column output schema matches the assignment specification exactly, and the dtype assertions confirm:
- `model_name`: object (string)
- `forecast_created_at`: datetime64[ns]
- `target_date`: datetime64[ns]
- `he`: int8 (range 1-24)
- `bus_id`: string (verified via spot-check; pandas-categorical-as-string would also pass)
- `zone_id`: category (consistent with notebook 03)
- `predict_pd`: float32

**Spot-check on the canonical first row** (bus `36POD_138KV_1` in FWES at 2025-01-01 HE 1) lets us compare zone-direct against the three notebook 03 baselines:

| Model | predict_pd at this slot (MW) | Interpretation |
|---|---|---|
| prev_week | 22.92 | Bus value 7 days earlier (2024-12-25 00:00) |
| hist_avg | 22.89 | Bus's training-period (hour=0, dow=Wed) average |
| prev_year | 64.54 | Bus value 1 year earlier (2024-01-02 00:00) — note the calendar misalignment |
| **zone_direct_lgbm (nextday)** | **29.40** | LightGBM zone forecast × bus's hour-0 share |
| **zone_direct_lgbm (nextmonth)** | **28.13** | Same approach with longer-horizon zone forecast |

Our predictions sit between the recent-history baselines (~23 MW) and the year-ago baseline (~65 MW), closer to the recent-history values. This is methodologically reasonable for an early-January 2025 forecast: recent 2024 data is the most informative reference, and our LightGBM model learned to weight it accordingly through the lag and trailing-mean features.

**The full picture of notebook 04's outputs:**
- 16 best-params JSON files in `data/processed/zone_models/` (v1, canonical)
- 16 v2 best-params JSON files (documentation of the failed mitigation experiment)
- 1 `bus_shares.parquet` (the disaggregation key)
- 2 `zone_forecasts_{nextday,nextmonth}.parquet` (intermediate zone-level forecasts, for Q5 in notebook 06)
- 2 final bus-level forecast parquets (the assignment deliverables)

In [19]:
"""
Final verification: confirm both zone-direct LightGBM forecast files are present
and well-formed.

For each file, check:
  - File exists at the expected path
  - Schema matches the assignment requirement (7 columns in correct order)
  - Row count is 32,427,554 (matches 2025 target set, excluding 2025-12-04)
  - No NaN values in predict_pd
  - forecast_created_at distinct value count matches expectation:
      next-day:   364 distinct values (one per target day, minus 2025-12-04)
      next-month: 12 distinct values  (one per target month)
  - HE range is 1-24
  - target_date covers all of 2025 minus 2025-12-04
  - Sample row inspection

This cell does not modify any data — it only inspects what was written.
"""

print("=" * 80)
print("Final verification — notebook 04 outputs")
print("=" * 80)

expected_files = [
    ("forecast_zone_direct_lgbm_nextday.parquet",   "zone_direct_lgbm_nextday",   364, "previous_day"),
    ("forecast_zone_direct_lgbm_nextmonth.parquet", "zone_direct_lgbm_nextmonth", 12,  "first_of_previous_month"),
]

EXPECTED_SCHEMA = ["model_name", "forecast_created_at", "target_date", "he",
                   "bus_id", "zone_id", "predict_pd"]
EXPECTED_ROWS = 32_427_554

summary_rows = []
for filename, expected_model_name, expected_fc_at_count, fc_at_strategy in expected_files:
    path = FORECASTS_DIR / filename
    print(f"\n{'─' * 80}")
    print(f"Checking: {filename}")
    print(f"{'─' * 80}")

    assert path.exists(), f"  ✗ Missing: {filename}"
    size_mb = path.stat().st_size / 1024**2
    print(f"  ✓ File present ({size_mb:.1f} MB)")

    df = pd.read_parquet(path)

    assert list(df.columns) == EXPECTED_SCHEMA, (
        f"  ✗ Schema mismatch: {list(df.columns)} != {EXPECTED_SCHEMA}"
    )
    print(f"  ✓ Schema: 7 columns in correct order")

    assert len(df) == EXPECTED_ROWS, f"  ✗ Row count {len(df):,} != {EXPECTED_ROWS:,}"
    print(f"  ✓ Row count: {len(df):,}")

    n_nan = df["predict_pd"].isna().sum()
    assert n_nan == 0, f"  ✗ Found {n_nan} NaN values in predict_pd"
    print(f"  ✓ No NaN values in predict_pd")

    unique_model_names = df["model_name"].unique()
    assert len(unique_model_names) == 1 and unique_model_names[0] == expected_model_name, (
        f"  ✗ model_name mismatch: got {unique_model_names}, expected '{expected_model_name}'"
    )
    print(f"  ✓ model_name: {expected_model_name}")

    n_fc_at = df["forecast_created_at"].nunique()
    assert n_fc_at == expected_fc_at_count, (
        f"  ✗ forecast_created_at has {n_fc_at} unique values, expected {expected_fc_at_count}"
    )
    print(f"  ✓ forecast_created_at: {n_fc_at} unique values (task convention: {fc_at_strategy})")

    # Verify 2025-12-04 is absent from next-day target_dates
    if fc_at_strategy == "previous_day":
        n_dec_4 = (df["target_date"] == pd.Timestamp("2025-12-04")).sum()
        assert n_dec_4 == 0, f"  ✗ Found {n_dec_4} rows for missing date 2025-12-04"
        print(f"  ✓ No rows for 2025-12-04 (correctly excluded)")

    he_min, he_max = df["he"].min(), df["he"].max()
    assert he_min == 1 and he_max == 24, f"  ✗ HE range {he_min}-{he_max} != 1-24"
    print(f"  ✓ HE range: {he_min}-{he_max}")

    td_min, td_max = df["target_date"].min(), df["target_date"].max()
    print(f"  ✓ target_date range: {td_min.date()} to {td_max.date()}")

    print(f"  ✓ predict_pd: mean={df['predict_pd'].mean():.2f}, "
          f"median={df['predict_pd'].median():.2f}, "
          f"max={df['predict_pd'].max():.2f}, "
          f"min={df['predict_pd'].min():.4f}")

    # Sample row
    print(f"  ✓ Sample row (first): "
          f"{df.iloc[0]['bus_id']} | {df.iloc[0]['zone_id']} | "
          f"target={df.iloc[0]['target_date'].date()} HE{df.iloc[0]['he']} | "
          f"predict_pd={df.iloc[0]['predict_pd']:.2f}")

    summary_rows.append({
        "file": filename,
        "size_mb": round(size_mb, 1),
        "rows": len(df),
        "model_name": expected_model_name,
        "fc_at_count": n_fc_at,
        "mean_predict_pd": round(df["predict_pd"].mean(), 2),
    })

    del df
    gc.collect()

# Summary
print("\n" + "=" * 80)
print("Summary of notebook 04 final outputs")
print("=" * 80)
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

print(f"\nAll files written to: {FORECASTS_DIR.resolve()}")
print(f"Total size: {sum(r['size_mb'] for r in summary_rows):.1f} MB")
print(f"Total rows: {sum(r['rows'] for r in summary_rows):,}")
print("\n✓ Notebook 04 complete — zone-direct LightGBM forecasts ready for evaluation in notebook 06")

Final verification — notebook 04 outputs

────────────────────────────────────────────────────────────────────────────────
Checking: forecast_zone_direct_lgbm_nextday.parquet
────────────────────────────────────────────────────────────────────────────────
  ✓ File present (141.2 MB)
  ✓ Schema: 7 columns in correct order
  ✓ Row count: 32,427,554
  ✓ No NaN values in predict_pd
  ✓ model_name: zone_direct_lgbm_nextday
  ✓ forecast_created_at: 364 unique values (task convention: previous_day)
  ✓ No rows for 2025-12-04 (correctly excluded)
  ✓ HE range: 1-24
  ✓ target_date range: 2025-01-01 to 2025-12-31
  ✓ predict_pd: mean=14.00, median=8.22, max=727.55, min=0.0000
  ✓ Sample row (first): 36POD_138KV_1 | FWES | target=2025-01-01 HE1 | predict_pd=29.40

────────────────────────────────────────────────────────────────────────────────
Checking: forecast_zone_direct_lgbm_nextmonth.parquet
────────────────────────────────────────────────────────────────────────────────
  ✓ File present (1

## Notebook 04 — complete

Both zone-direct LightGBM forecast files have been written to `data/processed/forecasts/` and verified against the assignment's required output schema.

| File | Size | Rows | model_name |
|---|---|---|---|
| `forecast_zone_direct_lgbm_nextday.parquet` | 141.2 MB | 32,427,554 | `zone_direct_lgbm_nextday` |
| `forecast_zone_direct_lgbm_nextmonth.parquet` | 120.5 MB | 32,427,554 | `zone_direct_lgbm_nextmonth` |
| **Total** | **261.7 MB** | **64,855,108** | |

**Pipeline summary:**

1. Aggregated 130.98M bus-level rows from notebook 02's features into 280K zone-hour rows across 8 zones
2. Built 23 model features for next-day and 20 for next-month, on the zone-aggregated series
3. Ran Optuna v1: 16 searches × 15 trials = 240 hyperparameter trials in 4.9 minutes
4. Diagnosed apparent failure on FWES/NOTH growth zones via prev_week sanity check
5. Attempted `recent_trend_ratio` mitigation (v2 search); the feature failed to improve performance on the failing zones
6. Proceeded with v1 hyperparameters
7. Retrained 16 final models on 2022-2024 with optimal `n_estimators` discovered via 2024 early-stopping (12 seconds total)
8. Generated zone-level 2025 forecasts (2,033 MW aggregate sum-of-squares RMSE for next-day = 3.88% of total zone mean)
9. Computed hour-of-day bus shares from 2022-2024 training-period data (4,166 non-cold-start buses + 42 cold-start buses + 341 sparse-training cells filled via zone-hour mean fallback)
10. Disaggregated zone forecasts to bus level with test-time renormalization (conservation property exact within float32 precision)
11. Wrote two final bus-level forecast files in the assignment's 7-column schema

**Key methodological findings carried forward to the report:**

- **Tree-based models cannot extrapolate beyond their training range at the leaf level.** Validation-stage diagnostics flagged FWES nextday as a 168%-of-baseline failure on the 2022-2023 → 2024 training-validation gap. The 2024-2025 gap was smaller, so the final-trained model (with 2024 in-distribution) generalized cleanly to 2025 (4.9% of zone mean).
- **Fixed-holdout validation on growing time series can overstate failure rates** when the train→val gap is the largest year-over-year jump in the series.
- **The recent_trend_ratio mitigation was attempted and documented to fail.** The feature carried insufficient signal for the actual distribution-shift problem; the principled feature would have been year-over-year growth ratio, but that feature has 100% NaN during the Optuna training window.
- **Test-time renormalization is required for top-down disaggregation.** Bus inventory varies across hours, so shares computed on the full training-period bus universe do not sum to 1.0 within each (zone, timestamp) of the test grid. Renormalization restores exact conservation.

**Ready for downstream notebooks:**
- Notebook 05a will produce the global-bus LightGBM comparator for Q1 (zone-direct vs global-bus)
- Notebook 06 will evaluate this notebook's forecasts alongside notebook 03's baselines and the upcoming 05 model variants
- The intermediate `zone_forecasts_*.parquet` files enable Q5 (sum-of-bus vs zone-direct at zone level) in notebook 06